# Credit Card Balance Feature Engineering

Here we turn monthly card snapshots into **one row per applicant in the previous-
application mapping**. We summarize each `SK_ID_PREV` first, then aggregate the
mapped contracts by `SK_ID_CURR`.

هنجمع الشهور لكل عقد كارت الأول، وبعد كده نجمع عقود الكروت لكل عميل.

We follow the small cells, named Pandas aggregations, English/Arabic explanations,
and Parquet checks in `01_bureau.ipynb` and the three following feature notebooks.
The decisions come from `../eda/06_credit_card_balance_eda.ipynb` and are checked
again against the CSV. No application outcome or `TARGET` is loaded.

| Grain | Meaning | المعنى بالعربي |
|---|---|---|
| Monthly row | One contract at one relative month | لقطة شهرية لعقد كارت |
| Contract | One `SK_ID_PREV` with its observed history | العقد السابق وسجله المتاح |
| Applicant | One mapped `SK_ID_CURR`, possibly several cards | العميل بعد تجميع عقوده |


## 1. Imports and Paths

`REPO_ROOT`, `DATA_RAW`, and `DATA_INTERIM` follow the Bureau convention. Searching
the current directory and its parents lets this notebook run from either the
project root or its own directory. SHA-256 fingerprints will verify raw contents.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import hashlib
import gc
from IPython.display import display, Markdown


In [2]:
# Locate the project without a personal username, drive, or fixed notebook depth.
REPO_ROOT = next(
    (folder for folder in [Path.cwd(), *Path.cwd().parents]
     if (folder / "data/raw/home-credit-default-risk").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError(
        "Open this notebook inside the project and extract the Kaggle CSVs "
        "into data/raw/home-credit-default-risk/."
    )
DATA_RAW = REPO_ROOT / "data/raw/home-credit-default-risk"
DATA_INTERIM = REPO_ROOT / "data/interim"
DATA_PROCESSED = REPO_ROOT / "data/processed"
CARD_PATH = DATA_RAW / "credit_card_balance.csv"
PREV_PATH = DATA_RAW / "previous_application.csv"
OUTPUT_PATH = DATA_INTERIM / "credit_card_features.parquet"
raw_sha256_before = {}
for path in [CARD_PATH, PREV_PATH]:
    with path.open("rb") as raw_file:
        raw_sha256_before[path] = hashlib.file_digest(raw_file, "sha256").hexdigest()
print("Project root:", REPO_ROOT)


Project root: e:\Depi Notes Data Engineering\DEPI project R5\ProJect


## 2. Load the Required Datasets

We keep monetary precision as `float64`, compact IDs/months, and categorical
statuses. DPD uses floating-point storage to preserve missing values, with observed
days validated as nonnegative integers. The full source schema is inspected before
we release unused drawing/maturity count columns.

Only the two mapping IDs are read from `previous_application.csv`. The actual card
CSV contains `SK_ID_CURR`; we audit it, then remove it. Feature applicant IDs always
come from the previous mapping. `AMT_RECIVABLE` keeps its original source spelling.
`AMT_TRANS_TOTAL` is absent and no substitute is created.


In [3]:
source_columns = pd.read_csv(CARD_PATH, nrows=0).columns.tolist()
MONEY_FIELDS = {
    "AMT_BALANCE": "balance", "AMT_CREDIT_LIMIT_ACTUAL": "credit_limit",
    "AMT_RECEIVABLE_PRINCIPAL": "principal_receivable", "AMT_RECIVABLE": "recivable",
    "AMT_TOTAL_RECEIVABLE": "total_receivable", "AMT_PAYMENT_CURRENT": "current_payment",
    "AMT_PAYMENT_TOTAL_CURRENT": "total_payment", "AMT_INST_MIN_REGULARITY": "minimum_instalment",
    "AMT_DRAWINGS_CURRENT": "drawings"
}
DRAWING_CATEGORIES = ["AMT_DRAWINGS_ATM_CURRENT", "AMT_DRAWINGS_POS_CURRENT", "AMT_DRAWINGS_OTHER_CURRENT"]
required_columns = ["SK_ID_PREV", "MONTHS_BALANCE", "NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF"] + list(MONEY_FIELDS) + DRAWING_CATEGORIES
assert set(required_columns).issubset(source_columns)
assert "AMT_TRANS_TOTAL" not in source_columns, "Inspect the changed source before defining transaction features."
has_source_applicant_id = "SK_ID_CURR" in source_columns
DTYPES = {name: "float64" for name in source_columns if name.startswith(("AMT_", "CNT_"))}
DTYPES.update({"SK_ID_PREV": "int32", "MONTHS_BALANCE": "int16", "NAME_CONTRACT_STATUS": "category", "SK_DPD": "float64", "SK_DPD_DEF": "float64"})
if has_source_applicant_id:
    DTYPES["SK_ID_CURR"] = "int32"
card = pd.read_csv(CARD_PATH, dtype=DTYPES, keep_default_na=False, na_values=[""])
previous_mapping = pd.read_csv(PREV_PATH, usecols=["SK_ID_PREV", "SK_ID_CURR"], dtype={"SK_ID_PREV": "int32", "SK_ID_CURR": "int32"})


## 3. Basic Source Inspection

The EDA reported about 3.84 million rows and 104,307 contracts. We calculate the
dimensions and missingness from this execution rather than hardcoding them.
Zero, missing, and negative amounts are different states; negative balances or
receivables may reflect corrections/refunds and remain in all amount statistics.


In [4]:
source_shape = card.shape
n_source_contracts = card["SK_ID_PREV"].nunique()
print("Source dimensions:", source_shape)
print("Unique previous contracts:", n_source_contracts)
print("Source has SK_ID_CURR:", has_source_applicant_id)
print("AMT_RECIVABLE preserved; AMT_TRANS_TOTAL absent.")
card.head()


Source dimensions: (3840312, 23)
Unique previous contracts: 104307
Source has SK_ID_CURR: True
AMT_RECIVABLE preserved; AMT_TRANS_TOTAL absent.


,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,...,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000.0,0.0,877.5,0.0,877.5,1700.325,...,0.000,0.000,0.0,1.0,0.0,1.0,35.0,Active,0.0,0.0
1,2582071,363914,-1,63975.555,45000.0,2250.0,2250.0,0.0,0.0,2250.000,...,64875.555,64875.555,1.0,1.0,0.0,0.0,69.0,Active,0.0,0.0
2,1740877,371185,-7,31815.225,450000.0,0.0,0.0,0.0,0.0,2250.000,...,31460.085,31460.085,0.0,0.0,0.0,0.0,30.0,Active,0.0,0.0
3,1389973,337855,-4,236572.110,225000.0,2250.0,2250.0,0.0,0.0,11795.760,...,233048.970,233048.970,1.0,1.0,0.0,0.0,10.0,Active,0.0,0.0
4,1891521,126868,-1,453919.455,450000.0,0.0,11547.0,0.0,11547.0,22924.890,...,453919.455,453919.455,0.0,1.0,0.0,1.0,101.0,Active,0.0,0.0


In [5]:
card.info(memory_usage="deep")
display(card.isna().sum().rename("missing_source_rows").to_frame())
display(card["NAME_CONTRACT_STATUS"].value_counts(dropna=False))
money_inspection = pd.DataFrame({
    "observed": card[list(MONEY_FIELDS) + DRAWING_CATEGORIES].count(),
    "missing": card[list(MONEY_FIELDS) + DRAWING_CATEGORIES].isna().sum(),
    "zero": card[list(MONEY_FIELDS) + DRAWING_CATEGORIES].eq(0).sum(),
    "negative": card[list(MONEY_FIELDS) + DRAWING_CATEGORIES].lt(0).sum(),
    "minimum": card[list(MONEY_FIELDS) + DRAWING_CATEGORIES].min(),
    "maximum": card[list(MONEY_FIELDS) + DRAWING_CATEGORIES].max()
})
display(money_inspection)


<class 'pandas.DataFrame'>
RangeIndex: 3840312 entries, 0 to 3840311
Data columns (total 23 columns):
 #   Column                      Dtype   
---  ------                      -----   
 0   SK_ID_PREV                  int32   
 1   SK_ID_CURR                  int32   
 2   MONTHS_BALANCE              int16   
 3   AMT_BALANCE                 float64 
 4   AMT_CREDIT_LIMIT_ACTUAL     float64 
 5   AMT_DRAWINGS_ATM_CURRENT    float64 
 6   AMT_DRAWINGS_CURRENT        float64 
 7   AMT_DRAWINGS_OTHER_CURRENT  float64 
 8   AMT_DRAWINGS_POS_CURRENT    float64 
 9   AMT_INST_MIN_REGULARITY     float64 
 10  AMT_PAYMENT_CURRENT         float64 
 11  AMT_PAYMENT_TOTAL_CURRENT   float64 
 12  AMT_RECEIVABLE_PRINCIPAL    float64 
 13  AMT_RECIVABLE               float64 
 14  AMT_TOTAL_RECEIVABLE        float64 
 15  CNT_DRAWINGS_ATM_CURRENT    float64 
 16  CNT_DRAWINGS_CURRENT        float64 
 17  CNT_DRAWINGS_OTHER_CURRENT  float64 
 18  CNT_DRAWINGS_POS_CURRENT    float64 
 19  CNT_INSTALM

,missing_source_rows
SK_ID_PREV,0
SK_ID_CURR,0
MONTHS_BALANCE,0
AMT_BALANCE,0
AMT_CREDIT_LIMIT_ACTUAL,0
AMT_DRAWINGS_ATM_CURRENT,749816
AMT_DRAWINGS_CURRENT,0
AMT_DRAWINGS_OTHER_CURRENT,749816
AMT_DRAWINGS_POS_CURRENT,749816
AMT_INST_MIN_REGULARITY,305236


NAME_CONTRACT_STATUS
Active           3698436
Completed         128918
Signed             11058
Demand              1365
Sent proposal        513
Refused               17
Approved               5
Name: count, dtype: int64

,observed,missing,zero,negative,minimum,maximum
AMT_BALANCE,3840312,0,2156420,2345,-420250.185,1505902.185
AMT_CREDIT_LIMIT_ACTUAL,3840312,0,753823,0,0.000,1350000.000
AMT_RECEIVABLE_PRINCIPAL,3840312,0,2296167,2428,-423305.820,1472316.795
AMT_RECIVABLE,3840312,0,2113816,109338,-420250.185,1493338.185
AMT_TOTAL_RECEIVABLE,3840312,0,2113643,109330,-420250.185,1493338.185
AMT_PAYMENT_CURRENT,3072324,767988,390507,0,0.000,4289207.445
AMT_PAYMENT_TOTAL_CURRENT,3840312,0,2172223,0,0.000,4278315.690
AMT_INST_MIN_REGULARITY,3535076,305236,1928864,0,0.000,202882.005
AMT_DRAWINGS_CURRENT,3840312,0,3223443,3,-6211.620,2287098.315
AMT_DRAWINGS_ATM_CURRENT,3090496,749816,2665718,1,-6827.310,2115000.000


### Contract-Month Policy and Relative Time

The EDA found no repeated contract-month keys. For this reproducible feature
table, execution **stops for review** if any repeat appears; it does not choose
an arbitrary row or silently exclude an endpoint. Every source row is represented
after this check. Raw CSVs stay unchanged.

Months are negative offsets relative to the application. `-1` is more recent than
`-12`. Nonnegative offsets would require a source-availability decision first.


In [6]:
KEY = ["SK_ID_PREV", "MONTHS_BALANCE"]
assert card[KEY].notna().all().all()
assert card["MONTHS_BALANCE"].lt(0).all(), "Review observations at/after application time."
exact_duplicate_extras = int(card.duplicated().sum())
duplicate_key_extras = int(card.duplicated(KEY).sum())
print("Exact duplicate extras:", exact_duplicate_extras)
print("Repeated contract-month extras:", duplicate_key_extras)
assert duplicate_key_extras == 0, "Repeated contract-month keys require a documented resolution."
for source in list(MONEY_FIELDS) + DRAWING_CATEGORIES + ["SK_DPD", "SK_DPD_DEF"]:
    assert np.isfinite(card[source].dropna()).all(), f"Investigate nonfinite observed {source}."
for source in ["SK_DPD", "SK_DPD_DEF"]:
    assert card[source].dropna().ge(0).all()
    assert card[source].dropna().mod(1).eq(0).all()
validation_results = {"Source schema, IDs, relative time, unique monthly grain, and numeric domains": "PASS"}


Exact duplicate extras: 0
Repeated contract-month extras: 0


In [7]:
dictionary = pd.read_csv(DATA_RAW / "HomeCredit_columns_description.csv", encoding="cp1252")
display(dictionary.loc[dictionary["Table"].eq("credit_card_balance.csv"), ["Row", "Description"]])
del dictionary


,Row,Description
150,SK_ID_PREV,ID of previous credit in Home credit related t...
151,SK_ID_CURR,ID of loan in our sample
152,MONTHS_BALANCE,Month of balance relative to application date ...
153,AMT_BALANCE,Balance during the month of previous credit
154,AMT_CREDIT_LIMIT_ACTUAL,Credit card limit during the month of the prev...
155,AMT_DRAWINGS_ATM_CURRENT,Amount drawing at ATM during the month of the ...
156,AMT_DRAWINGS_CURRENT,Amount drawing during the month of the previou...
157,AMT_DRAWINGS_OTHER_CURRENT,Amount of other drawings during the month of t...
158,AMT_DRAWINGS_POS_CURRENT,Amount drawing or buying goods during the mont...
159,AMT_INST_MIN_REGULARITY,Minimal installment for this month of the prev...


## 4. Understand the Contract-to-Applicant Mapping

`monthly rows → SK_ID_PREV contract summary → previous_application mapping → SK_ID_CURR`.

Each previous ID must map to exactly one applicant. We audit the optional source
applicant ID at contract grain. Unmatched contracts are reported with their row
counts and later preserved in `unmatched_credit_card_contracts`; source applicant
IDs never bypass the required mapping.


In [8]:
assert previous_mapping.notna().all().all()
mapping_conflicts = int(previous_mapping.groupby("SK_ID_PREV")["SK_ID_CURR"].nunique().gt(1).sum())
assert mapping_conflicts == 0
assert previous_mapping["SK_ID_PREV"].is_unique
previous_index = previous_mapping.set_index("SK_ID_PREV")
source_records_per_contract = card.groupby("SK_ID_PREV").size()
source_ids = source_records_per_contract.index
matched_ids = source_ids.intersection(previous_index.index)
unmatched_ids = source_ids.difference(previous_index.index)
previous_without_card = previous_index.index.difference(source_ids)
n_matched_rows = int(source_records_per_contract.loc[matched_ids].sum())
n_unmatched_rows = int(source_records_per_contract.loc[unmatched_ids].sum())
n_applicants = previous_mapping["SK_ID_CURR"].nunique()
n_mapped_card_applicants = previous_index.loc[matched_ids, "SK_ID_CURR"].nunique()


In [9]:
n_source_applicants = None
if has_source_applicant_id:
    assert card["SK_ID_CURR"].notna().all()
    source_id_grouped = card.groupby("SK_ID_PREV")["SK_ID_CURR"]
    assert source_id_grouped.nunique().le(1).all()
    source_applicant_map = source_id_grouped.first()
    assert source_applicant_map.loc[matched_ids].eq(previous_index.loc[matched_ids, "SK_ID_CURR"]).all()
    n_source_applicants = card["SK_ID_CURR"].nunique()
    del source_id_grouped, source_applicant_map
linkage_summary = pd.Series({
    "Matched card contracts": len(matched_ids), "Unmatched card contracts": len(unmatched_ids),
    "Matched monthly rows": n_matched_rows, "Unmatched monthly rows": n_unmatched_rows,
    "Mapped card applicants": n_mapped_card_applicants, "Final mapping applicants": n_applicants
}, name="Count")
display(linkage_summary.to_frame())
assert len(matched_ids) + len(unmatched_ids) == n_source_contracts
assert n_matched_rows + n_unmatched_rows == source_shape[0]
assert len(matched_ids) + len(previous_without_card) == len(previous_mapping)
card = card[required_columns].copy()
validation_results["Unambiguous mapping, source applicant audit, and unmatched contract/row conservation"] = "PASS"


,Count
Matched card contracts,92935
Unmatched card contracts,11372
Matched monthly rows,2757496
Unmatched monthly rows,1082816
Mapped card applicants,92447
Final mapping applicants,338857


The final table includes all applicants in the previous mapping, as in the other
feature notebooks. Applicant history coverage uses **all previous IDs** as its
denominator. An application without card history may have been a different product
or never created a loan; it is not a confirmed card loan with missing reporting.

عدد العقود اللي مفيش لها سجل كارت هنا محسوب من كل الطلبات السابقة، مش من عقود كروت مؤكدة.


## 5. Calculate and Inspect Monthly Utilization

Utilization is `AMT_BALANCE / AMT_CREDIT_LIMIT_ACTUAL`, requiring finite observed
values and a **positive limit**. A balance of 1,200 against a limit of 1,000 gives
`1.2`, or 120%. Fees, interest, limit changes, or reporting cutoffs may explain
above-limit values; this source cannot identify the mechanism. We never cap them.

Negative balances produce signed negative ratios when the limit is positive.
Zero balance with a positive limit is genuine zero utilization. Zero/missing
limits produce missing utilization, not zero. Thresholds ≥80%, ≥90%, and ≥100%
are analytical choices; >100% is a separate strict above-limit condition.

الاستخدام = الرصيد ÷ الحد الائتماني الموجب. ممكن يتعدى 100%؛ مش هنقصه أو نعتبره خطأ تلقائيًا.


In [10]:
utilization_eligible = (
    card["AMT_CREDIT_LIMIT_ACTUAL"].gt(0)
    & np.isfinite(card["AMT_CREDIT_LIMIT_ACTUAL"]) & np.isfinite(card["AMT_BALANCE"])
)
card["utilization"] = card["AMT_BALANCE"] / card["AMT_CREDIT_LIMIT_ACTUAL"].where(utilization_eligible)
assert card["utilization"].notna().eq(utilization_eligible).all()
assert card.loc[card["AMT_CREDIT_LIMIT_ACTUAL"].eq(0), "utilization"].isna().all()
np.testing.assert_allclose(card.loc[utilization_eligible, "utilization"], card.loc[utilization_eligible, "AMT_BALANCE"] / card.loc[utilization_eligible, "AMT_CREDIT_LIMIT_ACTUAL"])
source_utilization_audit = {
    "observed_months": int(card["utilization"].count()),
    "over100_months": int(card["utilization"].gt(1).sum()),
    "ge80_months": int(card["utilization"].ge(.8).sum()),
    "ge90_months": int(card["utilization"].ge(.9).sum()),
    "ge100_months": int(card["utilization"].ge(1).sum()),
    "maximum": card["utilization"].max()
}
display(pd.Series(source_utilization_audit).to_frame("Source utilization"))
display(card.loc[card["utilization"].gt(1), ["SK_ID_PREV", "MONTHS_BALANCE", "AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "utilization"]].head())
validation_results["Monthly utilization eligibility, signed values, and above-limit ratios preserved"] = "PASS"
del utilization_eligible


,Source utilization
observed_months,3.086489e+06
over100_months,3.343620e+05
ge80_months,8.879370e+05
ge90_months,7.046190e+05
ge100_months,3.345510e+05
maximum,1.177751e+01


,SK_ID_PREV,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,utilization
1,2582071,-1,63975.555,45000.0,1.421679
3,1389973,-4,236572.110,225000.0,1.051432
4,1891521,-1,453919.455,450000.0,1.008710
7,2095912,-7,47962.125,45000.0,1.065825
19,1189095,-4,452774.700,450000.0,1.006166


## 6. Select Latest Contract Snapshots

We sort once by contract and increasing relative month. `groupby.tail(1)` takes
the **entire final observed row** for each contract. `groupby.last()` can skip
missing values column by column and mix different months, so we avoid it.

Latest is latest observed, not necessarily month −1 or today's state. Later
applicant latest-state totals combine each card's latest snapshot, which can be
from different months. They are not guaranteed to be synchronized current exposure.


In [11]:
card = card.sort_values(KEY, ignore_index=True)
grouped = card.groupby("SK_ID_PREV", sort=True, observed=True)
latest_snapshot = grouped.tail(1).set_index("SK_ID_PREV").sort_index()
print("Latest snapshots:", len(latest_snapshot))
display(latest_snapshot["NAME_CONTRACT_STATUS"].value_counts(dropna=False).rename("latest_contracts").to_frame())
assert latest_snapshot.index.is_unique
assert len(latest_snapshot) == n_source_contracts
np.testing.assert_array_equal(latest_snapshot["MONTHS_BALANCE"], grouped["MONTHS_BALANCE"].max())


Latest snapshots: 104307


,latest_contracts
NAME_CONTRACT_STATUS,
Active,90952
Completed,13295
Signed,38
Demand,22
Sent proposal,0
Refused,0
Approved,0


## 7. Engineer Contract-Level Features

### A. History Length and Coverage Gaps

`size` counts rows and `nunique` counts distinct monthly offsets. They agree
after our unique-key check. Inclusive span is `latest - oldest + 1`; gaps are
span minus observed months. Offsets `[-5, -3, -2]` give three observations, a
four-month span, and one missing month. A gap does not prove missed repayment.


In [12]:
contract_features = grouped.agg(
    monthly_records=("MONTHS_BALANCE", "size"), observed_months=("MONTHS_BALANCE", "nunique"),
    oldest_month=("MONTHS_BALANCE", "min"), latest_month=("MONTHS_BALANCE", "max")
)
contract_features["history_span_months"] = contract_features["latest_month"] - contract_features["oldest_month"] + 1
contract_features["gap_months"] = contract_features["history_span_months"] - contract_features["observed_months"]
contract_features["has_history_gap"] = contract_features["gap_months"].gt(0)
latest_columns = list(MONEY_FIELDS) + ["NAME_CONTRACT_STATUS", "SK_DPD", "SK_DPD_DEF", "utilization"]
contract_features = contract_features.join(latest_snapshot[latest_columns].add_prefix("latest_"), validate="one_to_one")
display(contract_features[["monthly_records", "observed_months", "history_span_months", "gap_months"]].describe())


,monthly_records,observed_months,history_span_months,gap_months
count,104307.000000,104307.000000,104307.000000,104307.000000
mean,36.817395,36.817395,36.818229,0.000834
std,33.362639,33.362639,33.363619,0.086969
min,1.000000,1.000000,1.000000,0.000000
25%,10.000000,10.000000,10.000000,0.000000
50%,21.000000,21.000000,21.000000,0.000000
75%,74.000000,74.000000,74.000000,0.000000
max,96.000000,96.000000,96.000000,12.000000


### B. Latest Status and Observed Status Changes

Latest status flags count contracts, not monthly rows. Active and Completed
remain separate; other known statuses share a count, while missing/XNA/XAP are
unknown. Status changes compare adjacent **observed snapshots** with known statuses
at both ends. A reporting gap can hide when the change occurred. We retain
evaluable-pair coverage; no pair gives an unknown change indicator.


In [13]:
known_latest_status = contract_features["latest_NAME_CONTRACT_STATUS"].notna() & ~contract_features["latest_NAME_CONTRACT_STATUS"].isin(["XNA", "XAP"])
contract_features["latest_is_active"] = contract_features["latest_NAME_CONTRACT_STATUS"].eq("Active")
contract_features["latest_is_completed"] = contract_features["latest_NAME_CONTRACT_STATUS"].eq("Completed")
contract_features["latest_is_unknown"] = ~known_latest_status
contract_features["latest_is_other"] = known_latest_status & ~contract_features["latest_NAME_CONTRACT_STATUS"].isin(["Active", "Completed"])
month_step = grouped["MONTHS_BALANCE"].diff()
prior_status = grouped["NAME_CONTRACT_STATUS"].shift()
known_status_pair = (
    month_step.notna() & card["NAME_CONTRACT_STATUS"].notna() & prior_status.notna()
    & ~card["NAME_CONTRACT_STATUS"].isin(["XNA", "XAP"]) & ~prior_status.isin(["XNA", "XAP"])
)
card["known_status_pair"] = known_status_pair
card["status_changed"] = known_status_pair & card["NAME_CONTRACT_STATUS"].ne(prior_status)
contract_features["status_evaluable_pairs"] = grouped["known_status_pair"].sum()
contract_features["status_change_count"] = grouped["status_changed"].sum()
contract_features["has_status_change"] = contract_features["status_change_count"].gt(0).astype("float32").mask(contract_features["status_evaluable_pairs"].eq(0))
source_status_changes = int(card.loc[card["status_changed"], "SK_ID_PREV"].nunique())
display(pd.crosstab(prior_status.loc[card["status_changed"]], card.loc[card["status_changed"], "NAME_CONTRACT_STATUS"]))
del card["known_status_pair"], card["status_changed"]
del prior_status, known_status_pair, known_latest_status


NAME_CONTRACT_STATUS,Active,Completed,Demand,Signed
NAME_CONTRACT_STATUS,,,,
Active,0,13235,19,0
Sent proposal,490,0,0,23
Signed,4875,17,0,0
Refused,16,0,0,1
Approved,3,0,0,2


## 8. Engineer Historical Monthly Behavior at Contract Grain

### A. Historical Amounts and Availability

Balances, limits, and receivables are **stocks**; summing them across months is
not exposure. We retain contract sums only to calculate record-weighted means
later, plus maxima where useful. Payments/drawings are reported monthly **flows**;
an observed drawing total is meaningful subject to source reporting conventions.

`count()` excludes missing values; `sum(min_count=1)` leaves an all-missing sum
missing. Negative observed amounts remain included. Source names are never renamed;
short labels below name derived features only.


In [14]:
source_money_audit = {}
for source, label in MONEY_FIELDS.items():
    contract_features[label + "_observed_months"] = grouped[source].count()
    contract_features[label + "_sum"] = grouped[source].sum(min_count=1)
    contract_features[label + "_max"] = grouped[source].max()
    contract_features[label + "_missing_months"] = contract_features["monthly_records"] - contract_features[label + "_observed_months"]
    source_money_audit[label] = {
        "observed": int(card[source].count()), "missing": int(card[source].isna().sum()),
        "negative": int(card[source].lt(0).sum()), "sum": card[source].sum(min_count=1)
    }


### B. Utilization Exposure and High-Utilization Months

The eligible-month count is the denominator for high-utilization shares. Missing
ratios are excluded, including zero-limit rows. Mean/min/max retain signed ratios
and values above 1. A contract with no eligible utilization has unknown burden,
not zero burden. Strict >100% flags are separate from the inclusive ≥100% share.


In [15]:
contract_features["utilization_observed_months"] = grouped["utilization"].count()
contract_features["utilization_sum"] = grouped["utilization"].sum(min_count=1)
contract_features["utilization_min"] = grouped["utilization"].min()
contract_features["utilization_max"] = grouped["utilization"].max()
for threshold, label in [(.8, "80"), (.9, "90"), (1, "100")]:
    card["is_high_utilization"] = card["utilization"].ge(threshold)
    contract_features["utilization_ge" + label + "_months"] = grouped["is_high_utilization"].sum()
card["is_over_limit"] = card["utilization"].gt(1)
contract_features["utilization_over100_months"] = grouped["is_over_limit"].sum()
contract_features["has_over100_utilization"] = contract_features["utilization_over100_months"].gt(0).astype("float32").mask(contract_features["utilization_observed_months"].eq(0))
contract_features["latest_over100_utilization"] = contract_features["latest_utilization"].gt(1).astype("float32").mask(contract_features["latest_utilization"].isna())
del card["is_high_utilization"], card["is_over_limit"]


### C. Ordinary and Adjusted Delinquency

`SK_DPD` is ordinary days past due. `SK_DPD_DEF` applies a tolerance that ignores
low debt amounts; its exact tolerance is not supplied. We calculate two separate
families with the same operations and different names.

Any positive month means observed lateness; repeated means at least two positive
months. Consecutive means a positive pair exactly one relative month apart with
both DPD values known. One positive month is not persistence, and repeated months
can describe aging arrears rather than separate late-payment episodes.
The ≥90-day flag is an analytical severity threshold, not an observed default label.


In [16]:
DPD_FAMILIES = {"SK_DPD": "dpd", "SK_DPD_DEF": "dpd_def"}
source_dpd_audit = {}
for source, prefix in DPD_FAMILIES.items():
    contract_features[prefix + "_observed_months"] = grouped[source].count()
    contract_features[prefix + "_missing_months"] = contract_features["monthly_records"] - contract_features[prefix + "_observed_months"]
    contract_features[prefix + "_sum"] = grouped[source].sum(min_count=1)
    contract_features[prefix + "_max"] = grouped[source].max()
    card["is_positive_dpd"] = card[source].gt(0)
    positive_counts = grouped["is_positive_dpd"].sum()
    no_known_dpd = contract_features[prefix + "_observed_months"].eq(0)
    contract_features[prefix + "_positive_months"] = positive_counts.astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_has_delinquency"] = positive_counts.gt(0).astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_has_repeated"] = positive_counts.ge(2).astype("float32").mask(no_known_dpd)
    contract_features[prefix + "_has_ge90"] = contract_features[prefix + "_max"].ge(90).astype("float32").mask(no_known_dpd)
    source_dpd_audit[prefix] = {
        "observed": int(card[source].count()), "missing": int(card[source].isna().sum()),
        "positive": int(card[source].gt(0).sum()), "sum": card[source].sum(min_count=1),
        "severe_months": int(card[source].ge(90).sum()),
        "severe_contracts": card.loc[card[source].ge(90), "SK_ID_PREV"].nunique()
    }
del card["is_positive_dpd"]


`groupby.shift()` retrieves the prior row inside each contract. We also check
the month difference equals one; adjacent rows separated by a reporting gap
are not consecutive months. No evaluable consecutive pair gives a missing
consecutive flag, rather than a zero flag implying adequate observation.


In [17]:
for source, prefix in DPD_FAMILIES.items():
    prior_dpd = grouped[source].shift()
    eligible_pair = month_step.eq(1) & card[source].notna() & prior_dpd.notna()
    card["dpd_pair_evaluable"] = eligible_pair
    card["dpd_positive_pair"] = eligible_pair & card[source].gt(0) & prior_dpd.gt(0)
    contract_features[prefix + "_consecutive_evaluable_pairs"] = grouped["dpd_pair_evaluable"].sum()
    contract_features[prefix + "_has_consecutive"] = (
        grouped["dpd_positive_pair"].sum().gt(0).astype("float32")
        .mask(contract_features[prefix + "_consecutive_evaluable_pairs"].eq(0))
    )
    source_dpd_audit[prefix]["consecutive_contracts"] = card.loc[card["dpd_positive_pair"], "SK_ID_PREV"].nunique()
del card["dpd_pair_evaluable"], card["dpd_positive_pair"]
del prior_dpd, eligible_pair, positive_counts, no_known_dpd


### D. Payments, Drawings, and Descriptive Payment Ratios

Current payment, total payment, and minimum installment retain separate statistics.
The EDA found total payment often **below** current payment; the names do not
establish ordering or interchangeability. Both source values remain unchanged.

We use **total current payment** for two same-month descriptive ratios: payment /
positive balance and payment / positive minimum. A finite nonnegative payment and
positive finite denominator are required. These ratios can exceed 1 and are not
verified repayment fractions or billing compliance. Exact statement alignment is unknown.


In [18]:
payments_observed_pair = card[["AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT"]].notna().all(axis=1)
payment_scope_difference = card["AMT_PAYMENT_TOTAL_CURRENT"] - card["AMT_PAYMENT_CURRENT"]
payment_scope_audit = pd.Series({
    "Both payment fields observed": int(payments_observed_pair.sum()),
    "Total payment below current payment": int((payments_observed_pair & payment_scope_difference.lt(-1e-6)).sum()),
    "Total payment above current payment": int((payments_observed_pair & payment_scope_difference.gt(1e-6)).sum())
}, name="Monthly rows")
display(payment_scope_audit.to_frame())
PAYMENT_RATIO_FIELDS = {"AMT_BALANCE": "total_payment_to_balance", "AMT_INST_MIN_REGULARITY": "total_payment_to_minimum"}
for denominator, label in PAYMENT_RATIO_FIELDS.items():
    eligible = card[denominator].gt(0) & np.isfinite(card[denominator]) & card["AMT_PAYMENT_TOTAL_CURRENT"].ge(0) & np.isfinite(card["AMT_PAYMENT_TOTAL_CURRENT"])
    card["payment_ratio"] = card["AMT_PAYMENT_TOTAL_CURRENT"].where(eligible) / card[denominator].where(eligible)
    contract_features[label + "_observed_months"] = grouped["payment_ratio"].count()
    contract_features[label + "_sum"] = grouped["payment_ratio"].sum(min_count=1)
    assert card["payment_ratio"].notna().eq(eligible).all()
    assert not np.isinf(card["payment_ratio"].dropna()).any()
card["zero_drawings"] = card["AMT_DRAWINGS_CURRENT"].eq(0)
contract_features["zero_drawing_months"] = grouped["zero_drawings"].sum()
del card["payment_ratio"], card["zero_drawings"]
del payments_observed_pair, payment_scope_difference, eligible


,Monthly rows
Both payment fields observed,3072324
Total payment below current payment,1187232
Total payment above current payment,0


### E. Drawing Categories and Negative Ledger Values

Total drawings can be observed even when ATM/POS/other categories are missing.
We keep their availability separate; missing categories are not filled with zero.
A category sum requires all three components. Source total drawings stay separate
because the EDA found some complete rows where total and category sum disagree.
We do not construct a transaction total or replace the reported total with this sum.

Negative balance and negative-receivable flags detect signed ledger observations,
not confirmed errors or delinquency. The receivable flag counts a row once if
**any** of the three receivable fields is negative.


In [19]:
card["complete_drawing_categories"] = card[DRAWING_CATEGORIES].notna().all(axis=1)
contract_features["drawing_category_observed_months"] = grouped["complete_drawing_categories"].sum()
contract_features["drawing_category_missing_months"] = contract_features["monthly_records"] - contract_features["drawing_category_observed_months"]
category_sum = card[DRAWING_CATEGORIES].sum(axis=1, min_count=len(DRAWING_CATEGORIES))
drawing_difference = card["AMT_DRAWINGS_CURRENT"] - category_sum
source_drawing_audit = {
    "complete_comparable_rows": int(drawing_difference.count()),
    "amount_mismatch_rows": int(drawing_difference.abs().gt(1e-6).sum())
}
display(pd.Series(source_drawing_audit).to_frame("Drawing source audit"))
RECEIVABLE_FIELDS = ["AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE"]
for label, flag in {
    "negative_balance": card["AMT_BALANCE"].lt(0),
    "negative_receivable": card[RECEIVABLE_FIELDS].lt(0).any(axis=1),
    "zero_credit_limit": card["AMT_CREDIT_LIMIT_ACTUAL"].eq(0)
}.items():
    card["quality_flag"] = flag
    contract_features[label + "_months"] = grouped["quality_flag"].sum()
    contract_features[label + "_has_observation"] = contract_features[label + "_months"].gt(0)
del card["quality_flag"], card["complete_drawing_categories"]
del category_sum, drawing_difference, flag


,Drawing source audit
complete_comparable_rows,3090496
amount_mismatch_rows,7150


### F. Differences Between Ordinary and Adjusted DPD

Only months with both DPD values known are comparable. Disagreement is recorded
as a descriptive availability/definition feature, not automatically treated as
an error. Adjusted DPD greater than ordinary is reported for review without correction.


In [20]:
card["dpd_comparable"] = card[["SK_DPD", "SK_DPD_DEF"]].notna().all(axis=1)
card["dpd_disagreement"] = card["dpd_comparable"] & card["SK_DPD"].ne(card["SK_DPD_DEF"])
contract_features["dpd_comparable_months"] = grouped["dpd_comparable"].sum()
contract_features["dpd_disagreement_months"] = grouped["dpd_disagreement"].sum()
contract_features["has_dpd_disagreement"] = contract_features["dpd_disagreement_months"].gt(0).astype("float32").mask(contract_features["dpd_comparable_months"].eq(0))
source_dpd_disagreement_rows = int(card["dpd_disagreement"].sum())
source_adjusted_greater_rows = int((card["dpd_comparable"] & card["SK_DPD_DEF"].gt(card["SK_DPD"])).sum())
print("DPD disagreement rows:", source_dpd_disagreement_rows)
print("Adjusted DPD greater than ordinary:", source_adjusted_greater_rows)
del card["dpd_comparable"], card["dpd_disagreement"]


DPD disagreement rows: 64439
Adjusted DPD greater than ordinary: 0


### G. Recent Observations: Offsets −6 Through −1

We use the same six-offset window as the POS/CASH feature notebook: **−6 ≤ month ≤ −1**.
It is relative to each applicant's application, not a shared calendar period or
six guaranteed observations. Recent high utilization means **≥90%** among eligible
recent ratios. No eligible recent observation gives unknown recent behavior.

Recent DPD has separate ordinary/adjusted known-month denominators. The most
recent positive DPD offset is the maximum signed month over the **whole** history.
It remains missing when no positive observation is recorded.


In [21]:
RECENT_MIN_MONTH, RECENT_MAX_MONTH = -6, -1
recent_mask = card["MONTHS_BALANCE"].between(RECENT_MIN_MONTH, RECENT_MAX_MONTH)
card["recent_month"] = recent_mask
card["recent_utilization"] = card["utilization"].where(recent_mask)
card["recent_high90"] = recent_mask & card["utilization"].ge(.9)
contract_features["recent_records"] = grouped["recent_month"].sum()
contract_features["recent_utilization_observed_months"] = grouped["recent_utilization"].count()
contract_features["recent_utilization_sum"] = grouped["recent_utilization"].sum(min_count=1)
contract_features["recent_high90_months"] = grouped["recent_high90"].sum()
for source, prefix in DPD_FAMILIES.items():
    card["recent_dpd"] = card[source].where(recent_mask)
    card["recent_positive_dpd"] = recent_mask & card[source].gt(0)
    card["positive_month_offset"] = card["MONTHS_BALANCE"].where(card[source].gt(0))
    contract_features[prefix + "_recent_observed_months"] = grouped["recent_dpd"].count()
    contract_features[prefix + "_recent_positive_months"] = grouped["recent_positive_dpd"].sum()
    contract_features[prefix + "_last_positive_month"] = grouped["positive_month_offset"].max()
del card["recent_month"], card["recent_utilization"], card["recent_high90"]
del card["recent_dpd"], card["recent_positive_dpd"], card["positive_month_offset"]
del recent_mask


C:\Users\legon 5\AppData\Local\Temp\ipykernel_43156\1179215659.py:14: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  contract_features[prefix + "_recent_observed_months"] = grouped["recent_dpd"].count()
C:\Users\legon 5\AppData\Local\Temp\ipykernel_43156\1179215659.py:15: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  contract_features[prefix + "_recent_positive_months"] = grouped["recent_positive_dpd"].sum()
C:\Users\legon 5\AppData\Local\Temp\ipykernel_43156\1179215659.py:16: PerformanceWarning: DataFrame is highly fragmented.  

### Validate Contract Features Before Applicant Aggregation

We check one row per contract, monthly-count conservation, latest full snapshots,
and gaps independently using differences between adjacent offsets. Both DPD
families and all amount fields are checked against their own source values.


In [22]:
assert contract_features.index.is_unique
assert len(contract_features) == n_source_contracts
assert contract_features["monthly_records"].sum() == source_shape[0]
assert contract_features["monthly_records"].eq(contract_features["observed_months"]).all()
assert contract_features["gap_months"].ge(0).all()
card["gap_between_observations"] = month_step.sub(1).clip(lower=0).fillna(0)
np.testing.assert_array_equal(contract_features["gap_months"], grouped["gap_between_observations"].sum())
del card["gap_between_observations"]
expected_latest = card.loc[grouped["MONTHS_BALANCE"].idxmax()].set_index("SK_ID_PREV").sort_index()
pd.testing.assert_frame_equal(latest_snapshot, expected_latest, check_exact=True)
assert contract_features["has_status_change"].sum() == source_status_changes
assert contract_features["has_status_change"].isna().eq(contract_features["status_evaluable_pairs"].eq(0)).all()
validation_results["Contract grain, monthly counts, independent gaps, actual latest rows, and status-pair coverage"] = "PASS"
del expected_latest


In [23]:
for source, label in MONEY_FIELDS.items():
    audit = source_money_audit[label]
    assert contract_features[label + "_observed_months"].sum() == audit["observed"]
    assert contract_features[label + "_missing_months"].sum() == audit["missing"]
    np.testing.assert_allclose(contract_features[label + "_sum"].sum(min_count=1), audit["sum"], rtol=1e-12, equal_nan=True)
assert contract_features["utilization_observed_months"].sum() == source_utilization_audit["observed_months"]
for label in ["80", "90", "100"]:
    assert contract_features["utilization_ge" + label + "_months"].sum() == source_utilization_audit["ge" + label + "_months"]
assert contract_features["utilization_over100_months"].sum() == source_utilization_audit["over100_months"]
assert contract_features["utilization_max"].max() == source_utilization_audit["maximum"]
assert contract_features["negative_balance_months"].sum() == source_money_audit["balance"]["negative"]
assert contract_features["dpd_disagreement_months"].sum() == source_dpd_disagreement_rows
validation_results["Amount observations/sums, negative balances, uncapped utilization, and DPD comparison conservation"] = "PASS"


In [24]:
for source, prefix in DPD_FAMILIES.items():
    audit = source_dpd_audit[prefix]
    assert contract_features[prefix + "_observed_months"].sum() == audit["observed"]
    assert contract_features[prefix + "_missing_months"].sum() == audit["missing"]
    assert contract_features[prefix + "_positive_months"].sum() == audit["positive"]
    np.testing.assert_allclose(contract_features[prefix + "_sum"].sum(min_count=1), audit["sum"], equal_nan=True)
    assert contract_features[prefix + "_has_ge90"].sum() == audit["severe_contracts"]
    assert contract_features[prefix + "_has_consecutive"].sum() == audit["consecutive_contracts"]
    assert contract_features[prefix + "_has_consecutive"].isna().eq(contract_features[prefix + "_consecutive_evaluable_pairs"].eq(0)).all()
    positives = contract_features[prefix + "_positive_months"]
    assert (positives.isna() | positives.le(contract_features[prefix + "_observed_months"])).all()
    validation_results[source + " contract observation, positivity, severity, and consecutive checks"] = "PASS"


### Check Random and Boundary Contracts Directly Against Source Rows

We retain a reproducible sample plus gaps, negative balances/receivables, zero
limits, high utilization, missing payments, and single/stale histories. Small
arrays independently verify the aggregation without looping over every contract.


In [25]:
check_contract_ids = contract_features.sample(12, random_state=42).index.tolist()
for column in ["has_history_gap", "negative_balance_months", "negative_receivable_months", "zero_credit_limit_months", "has_over100_utilization", "dpd_has_repeated", "dpd_def_has_repeated", "has_status_change"]:
    check_contract_ids += contract_features.loc[contract_features[column].gt(0)].head(2).index.tolist()
for column in ["utilization_observed_months", "current_payment_observed_months", "recent_records", "observed_months"]:
    check_contract_ids += contract_features.loc[contract_features[column].le(1)].head(2).index.tolist()
check_contract_ids += contract_features.loc[contract_features["latest_AMT_PAYMENT_CURRENT"].isna()].head(2).index.tolist()
check_contract_ids += contract_features.loc[contract_features["latest_utilization"].isna()].head(2).index.tolist()
check_contract_ids = sorted(set(check_contract_ids))
check_contract_rows = card.loc[card["SK_ID_PREV"].isin(check_contract_ids)].copy()


In [26]:
for contract_id, records in check_contract_rows.groupby("SK_ID_PREV"):
    actual = contract_features.loc[contract_id]
    months = records["MONTHS_BALANCE"].to_numpy()
    expected_gaps = set(range(int(months.min()), int(months.max()) + 1)) - set(months)
    assert actual["gap_months"] == len(expected_gaps)
    latest_row = records.loc[records["MONTHS_BALANCE"].idxmax()]
    for source in list(MONEY_FIELDS) + ["SK_DPD", "SK_DPD_DEF", "utilization"]:
        np.testing.assert_allclose(actual["latest_" + source], latest_row[source], equal_nan=True)
    balances = records["AMT_BALANCE"].to_numpy()
    limits = records["AMT_CREDIT_LIMIT_ACTUAL"].to_numpy()
    eligible = np.isfinite(balances) & np.isfinite(limits) & (limits > 0)
    util = balances[eligible] / limits[eligible]
    assert actual["utilization_observed_months"] == len(util)
    np.testing.assert_allclose(actual["utilization_max"], util.max() if len(util) else np.nan, equal_nan=True)
    np.testing.assert_allclose(actual["utilization_min"], util.min() if len(util) else np.nan, equal_nan=True)
    for threshold, label in [(.8, "80"), (.9, "90"), (1, "100")]:
        assert actual["utilization_ge" + label + "_months"] == (util >= threshold).sum()
    for source, label in MONEY_FIELDS.items():
        assert actual[label + "_observed_months"] == records[source].count()
        assert actual[label + "_missing_months"] == records[source].isna().sum()
        np.testing.assert_allclose(actual[label + "_sum"], records[source].sum(min_count=1), rtol=1e-12, equal_nan=True)
validation_results["Direct contract source checks: latest values, history gaps, signed amounts, and utilization"] = "PASS"


In [27]:
for contract_id, records in check_contract_rows.groupby("SK_ID_PREV"):
    actual = contract_features.loc[contract_id]
    months = records["MONTHS_BALANCE"].to_numpy()
    recent = (months >= RECENT_MIN_MONTH) & (months <= RECENT_MAX_MONTH)
    status = records["NAME_CONTRACT_STATUS"].astype("string").to_numpy(na_value=None)
    known = records["NAME_CONTRACT_STATUS"].notna().to_numpy() & ~records["NAME_CONTRACT_STATUS"].isin(["XNA", "XAP"]).to_numpy()
    status_pairs = known[:-1] & known[1:]
    assert actual["status_change_count"] == (status_pairs & (status[:-1] != status[1:])).sum()
    for source, prefix in DPD_FAMILIES.items():
        days = records[source].to_numpy(dtype="float64")
        known_days = np.isfinite(days)
        positive = days > 0
        eligible_pair = (np.diff(months) == 1) & known_days[:-1] & known_days[1:]
        consecutive = eligible_pair & positive[:-1] & positive[1:]
        expected_flag = float(consecutive.any()) if eligible_pair.any() else np.nan
        np.testing.assert_allclose(actual[prefix + "_has_consecutive"], expected_flag, equal_nan=True)
        assert actual[prefix + "_recent_observed_months"] == (recent & known_days).sum()
        assert actual[prefix + "_recent_positive_months"] == (recent & positive).sum()
        np.testing.assert_allclose(actual[prefix + "_last_positive_month"], months[positive].max() if positive.any() else np.nan, equal_nan=True)
validation_results["Direct ordinary/adjusted consecutive DPD, status-change, and relative recent-window checks"] = "PASS"
print("Contracts checked directly:", len(check_contract_ids))


Contracts checked directly: 27


### Keep a Small Applicant Source Sample and Release Monthly Data

We retain all source rows for a few mapped applicants, including boundary cases
and applicants without card history. After contract/source checks, releasing the
full monthly table reduces memory for applicant aggregation.


In [28]:
check_applicant_ids = previous_index.reindex(check_contract_ids)["SK_ID_CURR"].dropna().astype("int32").tolist()
no_card_applicants = previous_mapping.loc[~previous_mapping["SK_ID_CURR"].isin(previous_index.loc[matched_ids, "SK_ID_CURR"]), "SK_ID_CURR"].drop_duplicates()
check_applicant_ids += no_card_applicants.head(2).tolist()
check_applicant_ids = sorted(set(check_applicant_ids))
check_previous_mapping = previous_mapping.loc[previous_mapping["SK_ID_CURR"].isin(check_applicant_ids)].copy()
check_applicant_rows = card.loc[card["SK_ID_PREV"].isin(check_previous_mapping["SK_ID_PREV"])].copy()
del grouped, card, latest_snapshot, check_contract_rows, month_step
gc.collect()


0

## 9. Aggregate Contract Features by Applicant

We map IDs at **contract grain**, retain unmatched summaries, then group matched
contracts. Only already-aggregated applicant tables are combined later.


In [29]:
for column in contract_features.select_dtypes(include="integer").columns:
    contract_features[column] = contract_features[column].astype("int32")
contract_features["SK_ID_CURR"] = contract_features.index.map(previous_index["SK_ID_CURR"]).astype("Int32")
contract_features = contract_features.reset_index()
unmatched_credit_card_contracts = contract_features.loc[contract_features["SK_ID_CURR"].isna()].copy()
mapped_contracts = contract_features.loc[contract_features["SK_ID_CURR"].notna()].copy()
mapped_contracts["SK_ID_CURR"] = mapped_contracts["SK_ID_CURR"].astype("int32")
assert contract_features["SK_ID_PREV"].is_unique
assert len(unmatched_credit_card_contracts) == len(unmatched_ids)
assert len(mapped_contracts) == len(matched_ids)
assert unmatched_credit_card_contracts["monthly_records"].sum() == n_unmatched_rows
assert mapped_contracts["monthly_records"].sum() == n_matched_rows
display(unmatched_credit_card_contracts[["SK_ID_PREV", "SK_ID_CURR", "monthly_records", "latest_month"]].head())
credit_card_grouped = mapped_contracts.groupby("SK_ID_CURR", sort=True)
mapped_applicant_index = credit_card_grouped.size().index


C:\Users\legon 5\AppData\Local\Temp\ipykernel_43156\3510890743.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  contract_features["SK_ID_CURR"] = contract_features.index.map(previous_index["SK_ID_CURR"]).astype("Int32")
C:\Users\legon 5\AppData\Local\Temp\ipykernel_43156\3510890743.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  contract_features = contract_features.reset_index()


,SK_ID_PREV,SK_ID_CURR,monthly_records,latest_month
9,1000096,<NA>,96,-1
29,1000344,<NA>,96,-1
36,1000455,<NA>,96,-1
42,1000514,<NA>,94,-3
43,1000515,<NA>,93,-4


### A. Contract Counts, Latest Status, and Coverage

`credit_card_contracts_with_history` is also the total identifiable **mapped card
contract count**; we do not add an identical feature under another name. Total
previous IDs and IDs without card history describe mapping coverage, not confirmed
card contracts that should have reported. Unmapped card counts remain in the audit.

Latest Active/Completed ratios use observed mapped cards, not monthly rows.
Status-change counts retain their evaluable-contract denominator. `join()` aligns
these summaries by the named applicant index; `reset_index()` later restores the ID column.


In [30]:
mapped_contracts["has_status_pairs"] = mapped_contracts["status_evaluable_pairs"].gt(0)
cc_count_features = previous_mapping.groupby("SK_ID_CURR").agg(
    credit_card_total_previous_contracts=("SK_ID_PREV", "size")
)
observed_counts = credit_card_grouped.agg(
    credit_card_contracts_with_history=("SK_ID_PREV", "size"),
    credit_card_latest_active_contracts=("latest_is_active", "sum"),
    credit_card_latest_completed_contracts=("latest_is_completed", "sum"),
    credit_card_latest_other_status_contracts=("latest_is_other", "sum"),
    credit_card_latest_unknown_status_contracts=("latest_is_unknown", "sum"),
    credit_card_status_change_evaluable_contracts=("has_status_pairs", "sum"),
    credit_card_contracts_with_status_changes=("has_status_change", "sum")
)
cc_count_features = cc_count_features.join(observed_counts, how="left", validate="one_to_one")
STATUS_COUNT_COLUMNS = ["credit_card_latest_active_contracts", "credit_card_latest_completed_contracts", "credit_card_latest_other_status_contracts", "credit_card_latest_unknown_status_contracts"]
COMPLETE_COUNT_COLUMNS = ["credit_card_contracts_with_history", "credit_card_status_change_evaluable_contracts"] + STATUS_COUNT_COLUMNS
cc_count_features[COMPLETE_COUNT_COLUMNS] = cc_count_features[COMPLETE_COUNT_COLUMNS].fillna(0).astype("int32")
cc_count_features["credit_card_previous_contracts_without_card_history"] = cc_count_features["credit_card_total_previous_contracts"] - cc_count_features["credit_card_contracts_with_history"]
cc_count_features["credit_card_contracts_with_status_changes"] = cc_count_features["credit_card_contracts_with_status_changes"].mask(cc_count_features["credit_card_status_change_evaluable_contracts"].eq(0))


In [31]:
observed_cards = cc_count_features["credit_card_contracts_with_history"].replace(0, np.nan)
cc_count_features["credit_card_active_contract_ratio"] = cc_count_features["credit_card_latest_active_contracts"] / observed_cards
cc_count_features["credit_card_completed_contract_ratio"] = cc_count_features["credit_card_latest_completed_contracts"] / observed_cards
cc_count_features["credit_card_history_coverage_ratio"] = cc_count_features["credit_card_contracts_with_history"] / cc_count_features["credit_card_total_previous_contracts"]
RATE_COLUMNS = ["credit_card_active_contract_ratio", "credit_card_completed_contract_ratio", "credit_card_history_coverage_ratio"]


### B. History Length and Gaps

History-length means give each observed card contract equal weight. Total monthly
records give the amount of evidence. Span statistics remain distinct from observed
months, and signed latest/oldest offsets retain recency and stale reporting.


In [32]:
cc_history_features = credit_card_grouped.agg(
    credit_card_monthly_records=("monthly_records", "sum"),
    credit_card_history_months_mean=("observed_months", "mean"),
    credit_card_history_months_min=("observed_months", "min"),
    credit_card_history_months_max=("observed_months", "max"),
    credit_card_history_span_months_mean=("history_span_months", "mean"),
    credit_card_history_span_months_max=("history_span_months", "max"),
    credit_card_contracts_with_history_gaps=("has_history_gap", "sum"),
    credit_card_gap_months=("gap_months", "sum"),
    credit_card_latest_observed_month=("latest_month", "max"),
    credit_card_oldest_observed_month=("oldest_month", "min")
)
COMPLETE_COUNT_COLUMNS += ["credit_card_monthly_records", "credit_card_contracts_with_history_gaps", "credit_card_gap_months"]


### C. Historical and Latest Utilization

The historical mean and threshold shares are **eligible-month weighted** across
contracts. Min/max keep negative and above-limit ratios. Latest mean/max use one
latest snapshot per card and only its valid ratio. We do not substitute an older
valid ratio for a latest zero-limit snapshot. Ever/ latest above-limit counts
are missing if their respective observation population is empty.


In [33]:
mapped_contracts["has_utilization"] = mapped_contracts["utilization_observed_months"].gt(0)
mapped_contracts["has_latest_utilization"] = mapped_contracts["latest_utilization"].notna()
cc_utilization_features = credit_card_grouped.agg(
    credit_card_utilization_observed_months=("utilization_observed_months", "sum"),
    credit_card_utilization_observed_contracts=("has_utilization", "sum"),
    credit_card_utilization_min=("utilization_min", "min"),
    credit_card_utilization_max=("utilization_max", "max"),
    credit_card_ever_over100_utilization_contracts=("has_over100_utilization", "sum"),
    credit_card_latest_utilization_observed_contracts=("has_latest_utilization", "sum"),
    credit_card_latest_utilization_mean=("latest_utilization", "mean"),
    credit_card_latest_utilization_max=("latest_utilization", "max"),
    credit_card_latest_over100_utilization_contracts=("latest_over100_utilization", "sum")
)
denominator = cc_utilization_features["credit_card_utilization_observed_months"].replace(0, np.nan)
cc_utilization_features["credit_card_utilization_mean"] = credit_card_grouped["utilization_sum"].sum(min_count=1) / denominator
for label in ["80", "90", "100"]:
    feature = "credit_card_utilization_ge" + label + "_rate"
    cc_utilization_features[feature] = credit_card_grouped["utilization_ge" + label + "_months"].sum() / denominator
    RATE_COLUMNS.append(feature)
cc_utilization_features["credit_card_ever_over100_utilization_contracts"] = cc_utilization_features["credit_card_ever_over100_utilization_contracts"].mask(denominator.isna())
cc_utilization_features["credit_card_latest_over100_utilization_contracts"] = cc_utilization_features["credit_card_latest_over100_utilization_contracts"].mask(cc_utilization_features["credit_card_latest_utilization_observed_contracts"].eq(0))
COMPLETE_COUNT_COLUMNS += ["credit_card_utilization_observed_months", "credit_card_utilization_observed_contracts", "credit_card_latest_utilization_observed_contracts"]


### D. Balances, Limits, and Receivables

Historical means are monthly-record weighted; maxima show the largest reported
balance/limit. Latest totals sum **known values from each card's last snapshot**.
These snapshots can be from different months and partial availability is possible;
the totals are observed latest reported amounts, not guaranteed synchronized exposure.
An all-missing latest total remains missing. Signed values stay unchanged.


In [34]:
STOCK_FIELDS = {source: MONEY_FIELDS[source] for source in ["AMT_BALANCE", "AMT_CREDIT_LIMIT_ACTUAL", "AMT_RECEIVABLE_PRINCIPAL", "AMT_RECIVABLE", "AMT_TOTAL_RECEIVABLE"]}
cc_balance_features = pd.DataFrame(index=mapped_applicant_index)
for source, label in STOCK_FIELDS.items():
    known_months = credit_card_grouped[label + "_observed_months"].sum().replace(0, np.nan)
    cc_balance_features["credit_card_" + label + "_mean"] = credit_card_grouped[label + "_sum"].sum(min_count=1) / known_months
    cc_balance_features["credit_card_latest_" + label + "_total"] = credit_card_grouped["latest_" + source].sum(min_count=1)
for label in ["balance", "credit_limit"]:
    cc_balance_features["credit_card_" + label + "_max"] = credit_card_grouped[label + "_max"].max()


### E. Payment and Drawing Behavior

Current payment, total payment, minimum installment, and drawings retain their
own observed-month denominators and monthly mean/max. Drawing total sums observed
monthly flows; zero-drawing share excludes missing drawing amounts.

Payment-ratio means average eligible monthly ratios, not payments divided by
summed monthly balances. They can exceed 1. Latest payment means average the
available last-snapshot values across cards, keeping missing latest fields missing.


In [35]:
FLOW_FIELDS = {source: MONEY_FIELDS[source] for source in ["AMT_PAYMENT_CURRENT", "AMT_PAYMENT_TOTAL_CURRENT", "AMT_INST_MIN_REGULARITY", "AMT_DRAWINGS_CURRENT"]}
cc_payment_features = pd.DataFrame(index=mapped_applicant_index)
for source, label in FLOW_FIELDS.items():
    observed = credit_card_grouped[label + "_observed_months"].sum()
    cc_payment_features["credit_card_" + label + "_observed_months"] = observed
    cc_payment_features["credit_card_" + label + "_mean"] = credit_card_grouped[label + "_sum"].sum(min_count=1) / observed.replace(0, np.nan)
    cc_payment_features["credit_card_" + label + "_max"] = credit_card_grouped[label + "_max"].max()
    COMPLETE_COUNT_COLUMNS.append("credit_card_" + label + "_observed_months")
cc_payment_features["credit_card_drawings_amount_total"] = credit_card_grouped["drawings_sum"].sum(min_count=1)
cc_payment_features["credit_card_zero_drawing_rate"] = credit_card_grouped["zero_drawing_months"].sum() / cc_payment_features["credit_card_drawings_observed_months"].replace(0, np.nan)
for label in PAYMENT_RATIO_FIELDS.values():
    observed = credit_card_grouped[label + "_observed_months"].sum()
    cc_payment_features["credit_card_" + label + "_observed_months"] = observed
    cc_payment_features["credit_card_" + label + "_mean"] = credit_card_grouped[label + "_sum"].sum(min_count=1) / observed.replace(0, np.nan)
    COMPLETE_COUNT_COLUMNS.append("credit_card_" + label + "_observed_months")
cc_payment_features["credit_card_latest_current_payment_mean"] = credit_card_grouped["latest_AMT_PAYMENT_CURRENT"].mean()
cc_payment_features["credit_card_latest_total_payment_mean"] = credit_card_grouped["latest_AMT_PAYMENT_TOTAL_CURRENT"].mean()
RATE_COLUMNS.append("credit_card_zero_drawing_rate")


### F. Separate Ordinary and Adjusted DPD Applicant Features

We sum contract flags to count affected cards. Historical DPD means/rates are
**known-month weighted** across contracts. Contract delinquency ratios use cards
with at least one known DPD observation. Consecutive counts use cards with an
evaluable pair. Latest mean/max use actual last snapshots, with their own availability.

Longer histories increase detection opportunity, as the EDA's history-length
comparison showed. Counts should be read alongside history and observed-month
features rather than treated as intrinsic borrower risk.


In [36]:
dpd_feature_groups = {}
for source, prefix in DPD_FAMILIES.items():
    mapped_contracts[prefix + "_has_observed"] = mapped_contracts[prefix + "_observed_months"].gt(0)
    mapped_contracts[prefix + "_has_consecutive_pairs"] = mapped_contracts[prefix + "_consecutive_evaluable_pairs"].gt(0)
    mapped_contracts[prefix + "_has_latest_observed"] = mapped_contracts["latest_" + source].notna()
    feature_prefix = "credit_card_" + prefix
    features = credit_card_grouped.agg(**{
        feature_prefix + "_observed_months": (prefix + "_observed_months", "sum"),
        feature_prefix + "_missing_months": (prefix + "_missing_months", "sum"),
        feature_prefix + "_observed_contracts": (prefix + "_has_observed", "sum"),
        feature_prefix + "_positive_months": (prefix + "_positive_months", "sum"),
        feature_prefix + "_delinquent_contracts": (prefix + "_has_delinquency", "sum"),
        feature_prefix + "_repeated_contracts": (prefix + "_has_repeated", "sum"),
        feature_prefix + "_consecutive_contracts": (prefix + "_has_consecutive", "sum"),
        feature_prefix + "_consecutive_evaluable_contracts": (prefix + "_has_consecutive_pairs", "sum"),
        feature_prefix + "_ge90_contracts": (prefix + "_has_ge90", "sum"),
        feature_prefix + "_max": (prefix + "_max", "max"),
        feature_prefix + "_latest_mean": ("latest_" + source, "mean"),
        feature_prefix + "_latest_max": ("latest_" + source, "max"),
        feature_prefix + "_latest_observed_contracts": (prefix + "_has_latest_observed", "sum")
    })
    denominator = features[feature_prefix + "_observed_months"].replace(0, np.nan)
    features[feature_prefix + "_mean"] = credit_card_grouped[prefix + "_sum"].sum(min_count=1) / denominator
    for suffix in ["_positive_months", "_delinquent_contracts", "_repeated_contracts", "_ge90_contracts"]:
        features[feature_prefix + suffix] = features[feature_prefix + suffix].mask(denominator.isna())
    features[feature_prefix + "_consecutive_contracts"] = features[feature_prefix + "_consecutive_contracts"].mask(features[feature_prefix + "_consecutive_evaluable_contracts"].eq(0))
    features[feature_prefix + "_positive_rate"] = features[feature_prefix + "_positive_months"] / denominator
    features[feature_prefix + "_contract_delinquency_ratio"] = features[feature_prefix + "_delinquent_contracts"] / features[feature_prefix + "_observed_contracts"].replace(0, np.nan)
    RATE_COLUMNS += [feature_prefix + "_positive_rate", feature_prefix + "_contract_delinquency_ratio"]
    COMPLETE_COUNT_COLUMNS += [feature_prefix + suffix for suffix in ["_observed_months", "_missing_months", "_observed_contracts", "_consecutive_evaluable_contracts", "_latest_observed_contracts"]]
    dpd_feature_groups[prefix] = features


### G. Recent Relative-Window Behavior and DPD Recency

Recent means/rates use eligible **contract-month observations** in −6 through −1.
The same month can appear once per card, so the record total is not a count of
distinct applicant months. Recent high utilization is ≥90%. No known window
observations gives missing rates/indicators. Global last-positive DPD month is
closer to zero when more recent and stays missing when no positive month is observed.

Latest utilization, balances, payments, and DPD are already in their latest-state
feature groups; they are not duplicated here.


In [37]:
mapped_contracts["has_recent_history"] = mapped_contracts["recent_records"].gt(0)
cc_recent_features = credit_card_grouped.agg(
    credit_card_recent_6m_monthly_records=("recent_records", "sum"),
    credit_card_recent_6m_observed_contracts=("has_recent_history", "sum"),
    credit_card_recent_6m_utilization_observed_months=("recent_utilization_observed_months", "sum")
)
recent_known = cc_recent_features["credit_card_recent_6m_utilization_observed_months"].replace(0, np.nan)
cc_recent_features["credit_card_recent_6m_utilization_mean"] = credit_card_grouped["recent_utilization_sum"].sum(min_count=1) / recent_known
recent_high = credit_card_grouped["recent_high90_months"].sum()
cc_recent_features["credit_card_recent_6m_utilization_ge90_rate"] = recent_high / recent_known
cc_recent_features["credit_card_has_recent_6m_high_utilization"] = recent_high.gt(0).astype("float32").mask(recent_known.isna())
for source, prefix in DPD_FAMILIES.items():
    feature_prefix = "credit_card_" + prefix
    observed = credit_card_grouped[prefix + "_recent_observed_months"].sum()
    positive = credit_card_grouped[prefix + "_recent_positive_months"].sum()
    cc_recent_features[feature_prefix + "_recent_6m_observed_months"] = observed
    cc_recent_features[feature_prefix + "_recent_6m_positive_months"] = positive.mask(observed.eq(0))
    cc_recent_features[feature_prefix + "_recent_6m_positive_rate"] = positive / observed.replace(0, np.nan)
    cc_recent_features[feature_prefix + "_has_recent_6m_delinquency"] = positive.gt(0).astype("float32").mask(observed.eq(0))
    cc_recent_features[feature_prefix + "_last_positive_month"] = credit_card_grouped[prefix + "_last_positive_month"].max()
    RATE_COLUMNS.append(feature_prefix + "_recent_6m_positive_rate")
    COMPLETE_COUNT_COLUMNS.append(feature_prefix + "_recent_6m_observed_months")
RATE_COLUMNS.append("credit_card_recent_6m_utilization_ge90_rate")
COMPLETE_COUNT_COLUMNS += ["credit_card_recent_6m_monthly_records", "credit_card_recent_6m_observed_contracts", "credit_card_recent_6m_utilization_observed_months"]


### H. Missingness and Data Availability

Missing amounts are counted separately from zero payments/drawings. Current-
payment histories with no known value remain identifiable, even when total payment
is observed. Category drawing availability differs from total drawing availability.
Negative counts describe observed ledger states, and zero limits explain some
undefined utilization. DPD disagreement has its own known-pair denominator.

These availability/detected-state counts are zero for an empty observed set;
that zero does not fill a missing amount or establish clean repayment history.


In [38]:
mapped_contracts["has_missing_current_payment"] = mapped_contracts["current_payment_missing_months"].gt(0)
mapped_contracts["has_no_current_payment"] = mapped_contracts["current_payment_observed_months"].eq(0)
mapped_contracts["latest_missing_current_payment"] = mapped_contracts["latest_AMT_PAYMENT_CURRENT"].isna()
mapped_contracts["has_no_complete_drawing_categories"] = mapped_contracts["drawing_category_observed_months"].eq(0)
cc_availability_features = credit_card_grouped.agg(
    credit_card_contracts_with_missing_current_payments=("has_missing_current_payment", "sum"),
    credit_card_contracts_without_current_payment_amounts=("has_no_current_payment", "sum"),
    credit_card_latest_missing_current_payment_contracts=("latest_missing_current_payment", "sum"),
    credit_card_drawing_category_observed_months=("drawing_category_observed_months", "sum"),
    credit_card_missing_drawing_category_months=("drawing_category_missing_months", "sum"),
    credit_card_contracts_without_complete_drawing_categories=("has_no_complete_drawing_categories", "sum"),
    credit_card_negative_balance_months=("negative_balance_months", "sum"),
    credit_card_negative_balance_contracts=("negative_balance_has_observation", "sum"),
    credit_card_negative_receivable_months=("negative_receivable_months", "sum"),
    credit_card_negative_receivable_contracts=("negative_receivable_has_observation", "sum"),
    credit_card_zero_credit_limit_months=("zero_credit_limit_months", "sum"),
    credit_card_zero_credit_limit_contracts=("zero_credit_limit_has_observation", "sum"),
    credit_card_dpd_comparable_months=("dpd_comparable_months", "sum"),
    credit_card_dpd_disagreement_months=("dpd_disagreement_months", "sum"),
    credit_card_dpd_disagreement_contracts=("has_dpd_disagreement", "sum")
)
for label in FLOW_FIELDS.values():
    cc_availability_features["credit_card_" + label + "_missing_months"] = credit_card_grouped[label + "_missing_months"].sum()
cc_availability_features["credit_card_utilization_missing_months"] = credit_card_grouped["monthly_records"].sum() - credit_card_grouped["utilization_observed_months"].sum()
COMPLETE_COUNT_COLUMNS += [column for column in cc_availability_features if column not in ["credit_card_dpd_disagreement_months", "credit_card_dpd_disagreement_contracts"]]
for feature in ["credit_card_dpd_disagreement_months", "credit_card_dpd_disagreement_contracts"]:
    cc_availability_features[feature] = cc_availability_features[feature].mask(cc_availability_features["credit_card_dpd_comparable_months"].eq(0))


## 10. Combine Applicant-Level Feature Groups

All groups now use an applicant index. `reset_index()` makes `SK_ID_CURR` an explicit
column again, and one-to-one merges reject duplicate IDs. The count table supplies
the full previous-mapping population. We fill only listed availability/count fields
for applicants with no card history; measured values, eligible event counts, and
rates remain unknown when the required observations are absent.


In [39]:
feature_groups = {
    "Contract counts, latest status, and coverage": cc_count_features.reset_index(),
    "History length and gaps": cc_history_features.reset_index(),
    "Historical and latest utilization": cc_utilization_features.reset_index(),
    "Balances, limits, and receivables": cc_balance_features.reset_index(),
    "Payments, drawings, and descriptive ratios": cc_payment_features.reset_index(),
    "Ordinary DPD": dpd_feature_groups["dpd"].reset_index(),
    "Adjusted DPD": dpd_feature_groups["dpd_def"].reset_index(),
    "Recent observations and DPD recency": cc_recent_features.reset_index(),
    "Missingness and observed ledger states": cc_availability_features.reset_index()
}
for name, group in feature_groups.items():
    assert group["SK_ID_CURR"].is_unique, name
    assert len(group) == (n_applicants if name == "Contract counts, latest status, and coverage" else n_mapped_card_applicants), name
credit_card_features = feature_groups["Contract counts, latest status, and coverage"].copy()
for group in list(feature_groups.values())[1:]:
    credit_card_features = credit_card_features.merge(group, on="SK_ID_CURR", how="left", validate="one_to_one")
credit_card_features = credit_card_features.sort_values("SK_ID_CURR").reset_index(drop=True)
no_card_history = credit_card_features["credit_card_contracts_with_history"].eq(0)
credit_card_features.loc[no_card_history, COMPLETE_COUNT_COLUMNS] = 0
for column in COMPLETE_COUNT_COLUMNS + ["credit_card_total_previous_contracts", "credit_card_previous_contracts_without_card_history"]:
    credit_card_features[column] = credit_card_features[column].astype("int32")
print("Shape:", credit_card_features.shape)
print("Applicants with mapped card history:", n_mapped_card_applicants)
print("Applicants without mapped card history:", int(no_card_history.sum()))
credit_card_features.head()


Shape: (338857, 136)
Applicants with mapped card history: 92447
Applicants without mapped card history: 246410


,SK_ID_CURR,credit_card_total_previous_contracts,credit_card_contracts_with_history,credit_card_latest_active_contracts,credit_card_latest_completed_contracts,credit_card_latest_other_status_contracts,credit_card_latest_unknown_status_contracts,credit_card_status_change_evaluable_contracts,credit_card_contracts_with_status_changes,credit_card_previous_contracts_without_card_history,...,credit_card_zero_credit_limit_months,credit_card_zero_credit_limit_contracts,credit_card_dpd_comparable_months,credit_card_dpd_disagreement_months,credit_card_dpd_disagreement_contracts,credit_card_current_payment_missing_months,credit_card_total_payment_missing_months,credit_card_minimum_instalment_missing_months,credit_card_drawings_missing_months,credit_card_utilization_missing_months
0,100001,1,0,0,0,0,0,0,NaN,1,...,0,0,0,NaN,NaN,0,0,0,0,0
1,100002,1,0,0,0,0,0,0,NaN,1,...,0,0,0,NaN,NaN,0,0,0,0,0
2,100003,3,0,0,0,0,0,0,NaN,3,...,0,0,0,NaN,NaN,0,0,0,0,0
3,100004,1,0,0,0,0,0,0,NaN,1,...,0,0,0,NaN,NaN,0,0,0,0,0
4,100005,2,0,0,0,0,0,0,NaN,2,...,0,0,0,NaN,NaN,0,0,0,0,0


## 11. Validation

We check exact applicant coverage, conserved mapped contracts/rows, latest status
counts, DPD flags, ratio formulas, and missingness. Direct source examples verify
historical weighting and latest snapshots independently of the contract summaries.


In [40]:
assert credit_card_features["SK_ID_CURR"].notna().all()
assert credit_card_features["SK_ID_CURR"].is_unique
assert len(credit_card_features) == n_applicants
np.testing.assert_array_equal(credit_card_features["SK_ID_CURR"], np.sort(previous_mapping["SK_ID_CURR"].unique()))
assert credit_card_features.columns.is_unique
assert all(column.startswith("credit_card_") for column in credit_card_features if column != "SK_ID_CURR")
assert "TARGET" not in credit_card_features.columns
assert credit_card_features["credit_card_contracts_with_history"].sum() == len(matched_ids)
assert credit_card_features["credit_card_monthly_records"].sum() == n_matched_rows
assert credit_card_features["credit_card_total_previous_contracts"].sum() == len(previous_mapping)
assert credit_card_features["credit_card_previous_contracts_without_card_history"].sum() == len(previous_without_card)
assert (credit_card_features["credit_card_contracts_with_history"] + credit_card_features["credit_card_previous_contracts_without_card_history"]).eq(credit_card_features["credit_card_total_previous_contracts"]).all()
assert credit_card_features[STATUS_COUNT_COLUMNS].sum(axis=1).eq(credit_card_features["credit_card_contracts_with_history"]).all()
validation_results["Unique final applicant grain, exact mapping coverage, and card/month count conservation"] = "PASS"


A cross-tabulation independently checks every applicant's latest status bucket.
These are one-per-contract snapshots, so a long history cannot inflate the counts.


In [41]:
status_bucket = mapped_contracts["latest_NAME_CONTRACT_STATUS"].astype("string").fillna("Unknown")
status_bucket = status_bucket.mask(status_bucket.isin(["XNA", "XAP"]), "Unknown")
status_bucket = status_bucket.where(status_bucket.isin(["Active", "Completed", "Unknown"]), "Other")
status_names = {"Active": "credit_card_latest_active_contracts", "Completed": "credit_card_latest_completed_contracts", "Other": "credit_card_latest_other_status_contracts", "Unknown": "credit_card_latest_unknown_status_contracts"}
expected_status = pd.crosstab(mapped_contracts["SK_ID_CURR"], status_bucket)
expected_status = expected_status.reindex(index=credit_card_features["SK_ID_CURR"], columns=list(status_names), fill_value=0).rename(columns=status_names)
pd.testing.assert_frame_equal(credit_card_features.set_index("SK_ID_CURR")[list(status_names.values())], expected_status, check_dtype=False, check_names=False)
assert credit_card_features["credit_card_contracts_with_status_changes"].sum() == mapped_contracts["has_status_change"].sum()
validation_results["Every applicant latest-status bucket and detected status changes conserved"] = "PASS"
del status_bucket, expected_status


In [42]:
for source, prefix in DPD_FAMILIES.items():
    feature_prefix = "credit_card_" + prefix
    for feature_suffix, contract_suffix in {
        "_positive_months": "_positive_months", "_delinquent_contracts": "_has_delinquency",
        "_repeated_contracts": "_has_repeated", "_consecutive_contracts": "_has_consecutive",
        "_ge90_contracts": "_has_ge90", "_recent_6m_positive_months": "_recent_positive_months"
    }.items():
        assert credit_card_features[feature_prefix + feature_suffix].sum() == mapped_contracts[prefix + contract_suffix].sum()
    positives = credit_card_features[feature_prefix + "_positive_months"]
    assert (positives.isna() | positives.le(credit_card_features[feature_prefix + "_observed_months"])).all()
    validation_results[source + " applicant positivity, recurrence, severity, and recent counts conserved"] = "PASS"
assert credit_card_features["credit_card_dpd_disagreement_months"].sum() == mapped_contracts["dpd_disagreement_months"].sum()
assert credit_card_features["credit_card_negative_balance_months"].sum() == mapped_contracts["negative_balance_months"].sum()
assert credit_card_features["credit_card_negative_receivable_months"].sum() == mapped_contracts["negative_receivable_months"].sum()
assert credit_card_features["credit_card_utilization_max"].max() == mapped_contracts["utilization_max"].max()
validation_results["Negative ledger observations and mapped utilization maxima retained"] = "PASS"


### Check Applicant Amounts and Latest State Directly Against Source Rows

For each retained applicant we fetch their previous IDs from the mapping, then
inspect all corresponding source rows. Independently sorting by month and taking
the final row per contract checks the snapshot totals/means, including missingness.
Applicants without card history are included explicitly.


In [43]:
check_features = credit_card_features.set_index("SK_ID_CURR").loc[check_applicant_ids].astype("float64")
for applicant_id in check_applicant_ids:
    previous_ids = check_previous_mapping.loc[check_previous_mapping["SK_ID_CURR"].eq(applicant_id), "SK_ID_PREV"]
    records = check_applicant_rows.loc[check_applicant_rows["SK_ID_PREV"].isin(previous_ids)]
    actual = check_features.loc[applicant_id]
    latest = records.sort_values("MONTHS_BALANCE").drop_duplicates("SK_ID_PREV", keep="last")
    assert actual["credit_card_contracts_with_history"] == records["SK_ID_PREV"].nunique()
    assert actual["credit_card_monthly_records"] == len(records)
    for source, label in MONEY_FIELDS.items():
        np.testing.assert_allclose(actual["credit_card_" + label + "_mean"], records[source].mean(), rtol=1e-12, equal_nan=True)
    for source, label in STOCK_FIELDS.items():
        np.testing.assert_allclose(actual["credit_card_latest_" + label + "_total"], latest[source].sum(min_count=1), rtol=1e-12, equal_nan=True)
    for source, label in FLOW_FIELDS.items():
        np.testing.assert_allclose(actual["credit_card_" + label + "_max"], records[source].max(), equal_nan=True)
        assert actual["credit_card_" + label + "_observed_months"] == records[source].count()
    latest_util = latest["AMT_BALANCE"] / latest["AMT_CREDIT_LIMIT_ACTUAL"].where(latest["AMT_CREDIT_LIMIT_ACTUAL"].gt(0))
    np.testing.assert_allclose(actual["credit_card_latest_utilization_mean"], latest_util.mean(), equal_nan=True)
    np.testing.assert_allclose(actual["credit_card_latest_utilization_max"], latest_util.max(), equal_nan=True)
    np.testing.assert_allclose(actual["credit_card_latest_current_payment_mean"], latest["AMT_PAYMENT_CURRENT"].mean(), equal_nan=True)
    np.testing.assert_allclose(actual["credit_card_latest_total_payment_mean"], latest["AMT_PAYMENT_TOTAL_CURRENT"].mean(), equal_nan=True)
    np.testing.assert_allclose(actual["credit_card_drawings_amount_total"], records["AMT_DRAWINGS_CURRENT"].sum(min_count=1), rtol=1e-12, equal_nan=True)
    recent = records.loc[records["MONTHS_BALANCE"].between(RECENT_MIN_MONTH, RECENT_MAX_MONTH)]
    np.testing.assert_allclose(actual["credit_card_recent_6m_utilization_mean"], recent["utilization"].mean(), equal_nan=True)
validation_results["Direct applicant source checks: weighted amounts, actual latest stock/payment/utilization, and recent means"] = "PASS"


In [44]:
for applicant_id in check_applicant_ids:
    previous_ids = check_previous_mapping.loc[check_previous_mapping["SK_ID_CURR"].eq(applicant_id), "SK_ID_PREV"]
    records = check_applicant_rows.loc[check_applicant_rows["SK_ID_PREV"].isin(previous_ids)]
    latest = records.sort_values("MONTHS_BALANCE").drop_duplicates("SK_ID_PREV", keep="last")
    actual = check_features.loc[applicant_id]
    for source, prefix in DPD_FAMILIES.items():
        feature_prefix = "credit_card_" + prefix
        observed = records[source].count()
        positive_records = records.loc[records[source].gt(0)]
        expected_rate = len(positive_records) / observed if observed else np.nan
        np.testing.assert_allclose(actual[feature_prefix + "_mean"], records[source].mean(), equal_nan=True)
        np.testing.assert_allclose(actual[feature_prefix + "_max"], records[source].max(), equal_nan=True)
        np.testing.assert_allclose(actual[feature_prefix + "_positive_rate"], expected_rate, equal_nan=True)
        np.testing.assert_allclose(actual[feature_prefix + "_delinquent_contracts"], positive_records["SK_ID_PREV"].nunique() if observed else np.nan, equal_nan=True)
        repeated = positive_records.groupby("SK_ID_PREV").size().ge(2).sum()
        np.testing.assert_allclose(actual[feature_prefix + "_repeated_contracts"], repeated if observed else np.nan, equal_nan=True)
        np.testing.assert_allclose(actual[feature_prefix + "_latest_mean"], latest[source].mean(), equal_nan=True)
        np.testing.assert_allclose(actual[feature_prefix + "_latest_max"], latest[source].max(), equal_nan=True)
        np.testing.assert_allclose(actual[feature_prefix + "_last_positive_month"], positive_records["MONTHS_BALANCE"].max(), equal_nan=True)
        recent = records.loc[records["MONTHS_BALANCE"].between(RECENT_MIN_MONTH, RECENT_MAX_MONTH)]
        recent_known = recent[source].count()
        expected_recent_rate = recent[source].gt(0).sum() / recent_known if recent_known else np.nan
        np.testing.assert_allclose(actual[feature_prefix + "_recent_6m_positive_rate"], expected_recent_rate, equal_nan=True)
validation_results["Independent ordinary/adjusted applicant delay, recurrence, latest state, and recency checks"] = "PASS"
print("Applicants checked directly:", len(check_applicant_ids))
del check_features, check_applicant_rows, check_previous_mapping


Applicants checked directly: 22


### Validate Rate Formulas and Numeric Limits

Frequency/coverage shares lie in 0–1, while utilization and payment ratios can
exceed 1. Signed utilization and balance statistics can be negative. We check
every rate against its exact known-observation denominator and check the whole
numeric table for infinities.


In [45]:
final_by_applicant = credit_card_features.set_index("SK_ID_CURR")
base_rate_definitions = {
    "credit_card_history_coverage_ratio": ("credit_card_contracts_with_history", "credit_card_total_previous_contracts"),
    "credit_card_active_contract_ratio": ("credit_card_latest_active_contracts", "credit_card_contracts_with_history"),
    "credit_card_completed_contract_ratio": ("credit_card_latest_completed_contracts", "credit_card_contracts_with_history")
}
contract_rate_definitions = {
    "credit_card_zero_drawing_rate": ("zero_drawing_months", "credit_card_drawings_observed_months"),
    "credit_card_recent_6m_utilization_ge90_rate": ("recent_high90_months", "credit_card_recent_6m_utilization_observed_months")
}
for label in ["80", "90", "100"]:
    contract_rate_definitions["credit_card_utilization_ge" + label + "_rate"] = ("utilization_ge" + label + "_months", "credit_card_utilization_observed_months")
for prefix in DPD_FAMILIES.values():
    feature_prefix = "credit_card_" + prefix
    contract_rate_definitions[feature_prefix + "_positive_rate"] = (prefix + "_positive_months", feature_prefix + "_observed_months")
    contract_rate_definitions[feature_prefix + "_contract_delinquency_ratio"] = (prefix + "_has_delinquency", feature_prefix + "_observed_contracts")
    contract_rate_definitions[feature_prefix + "_recent_6m_positive_rate"] = (prefix + "_recent_positive_months", feature_prefix + "_recent_6m_observed_months")
assert set(base_rate_definitions) | set(contract_rate_definitions) == set(RATE_COLUMNS)
for feature, (numerator, denominator) in base_rate_definitions.items():
    expected = final_by_applicant[numerator] / final_by_applicant[denominator].replace(0, np.nan)
    np.testing.assert_allclose(final_by_applicant[feature], expected, equal_nan=True)
for feature, (contract_column, denominator) in contract_rate_definitions.items():
    numerator = credit_card_grouped[contract_column].sum().reindex(final_by_applicant.index, fill_value=0)
    expected = numerator / final_by_applicant[denominator].replace(0, np.nan)
    np.testing.assert_allclose(final_by_applicant[feature], expected, equal_nan=True)
for feature, (_, denominator) in {**base_rate_definitions, **contract_rate_definitions}.items():
    assert final_by_applicant[feature].dropna().between(0, 1).all(), feature
    assert final_by_applicant[feature].isna().eq(final_by_applicant[denominator].eq(0)).all(), feature
assert not np.isinf(credit_card_features.to_numpy(dtype="float64", na_value=np.nan)).any()
validation_results["All documented eligible-rate formulas, 0–1 shares, and no unexpected infinities"] = "PASS"


### Missingness Must Match Data Availability

- No card history gives zero availability counts and missing measured history/state.
- Zero or missing limits give unknown utilization, not zero utilization.
- Missing payment/drawing fields remain unknown; total and category availability differ.
- No known DPD gives missing DPD measurements; no evaluable pair gives missing consecutive counts.
- No positive DPD gives a missing last-positive month, including clean observed histories.
- No eligible recent observation gives missing recent means, rates, and indicators.
- Latest statistics never borrow older nonmissing values to replace missing latest snapshots.

الصفر مع بيانات معروفة له معنى. لكن القيمة الناقصة معناها إن المعلومات اللازمة للحساب
مش متاحة، ومش هنحوّلها لصفر عشان الجدول يبان كامل.


In [46]:
missing_summary = pd.DataFrame({
    "missing_applicants": credit_card_features.isna().sum(),
    "missing_percent": (credit_card_features.isna().mean() * 100).round(2)
}).sort_values("missing_applicants", ascending=False)
display(missing_summary.loc[missing_summary["missing_applicants"].gt(0)])
assert credit_card_features[COMPLETE_COUNT_COLUMNS].notna().all().all()
for column in cc_history_features:
    if column not in COMPLETE_COUNT_COLUMNS:
        assert credit_card_features[column].isna().eq(no_card_history).all(), column
assert credit_card_features["credit_card_contracts_with_status_changes"].isna().eq(credit_card_features["credit_card_status_change_evaluable_contracts"].eq(0)).all()
for label in FLOW_FIELDS.values():
    observed = credit_card_features["credit_card_" + label + "_observed_months"]
    missing = credit_card_features["credit_card_" + label + "_missing_months"]
    assert (observed + missing).eq(credit_card_features["credit_card_monthly_records"]).all()
    for suffix in ["_mean", "_max"]:
        assert credit_card_features["credit_card_" + label + suffix].isna().eq(observed.eq(0)).all()
assert (credit_card_features["credit_card_utilization_observed_months"] + credit_card_features["credit_card_utilization_missing_months"]).eq(credit_card_features["credit_card_monthly_records"]).all()
assert (credit_card_features["credit_card_drawing_category_observed_months"] + credit_card_features["credit_card_missing_drawing_category_months"]).eq(credit_card_features["credit_card_monthly_records"]).all()


,missing_applicants,missing_percent
credit_card_dpd_def_last_positive_month,327703,96.71
credit_card_dpd_last_positive_month,324643,95.81
credit_card_total_payment_to_minimum_mean,278778,82.27
credit_card_total_payment_to_balance_mean,278392,82.16
credit_card_current_payment_max,276916,81.72
...,...,...
credit_card_dpd_def_delinquent_contracts,246410,72.72
credit_card_dpd_def_repeated_contracts,246410,72.72
credit_card_dpd_latest_max,246410,72.72
credit_card_dpd_mean,246410,72.72


In [47]:
for source, label in STOCK_FIELDS.items():
    observed_history = credit_card_grouped[label + "_observed_months"].sum().reindex(final_by_applicant.index, fill_value=0)
    observed_latest = credit_card_grouped["latest_" + source].count().reindex(final_by_applicant.index, fill_value=0)
    assert final_by_applicant["credit_card_" + label + "_mean"].isna().eq(observed_history.eq(0)).all()
    assert final_by_applicant["credit_card_latest_" + label + "_total"].isna().eq(observed_latest.eq(0)).all()
for feature, denominator in {
    "credit_card_utilization_mean": "credit_card_utilization_observed_months",
    "credit_card_utilization_min": "credit_card_utilization_observed_months",
    "credit_card_utilization_max": "credit_card_utilization_observed_months",
    "credit_card_ever_over100_utilization_contracts": "credit_card_utilization_observed_months",
    "credit_card_latest_utilization_mean": "credit_card_latest_utilization_observed_contracts",
    "credit_card_latest_utilization_max": "credit_card_latest_utilization_observed_contracts",
    "credit_card_latest_over100_utilization_contracts": "credit_card_latest_utilization_observed_contracts",
    "credit_card_recent_6m_utilization_mean": "credit_card_recent_6m_utilization_observed_months",
    "credit_card_has_recent_6m_high_utilization": "credit_card_recent_6m_utilization_observed_months",
    "credit_card_dpd_disagreement_months": "credit_card_dpd_comparable_months",
    "credit_card_dpd_disagreement_contracts": "credit_card_dpd_comparable_months"
}.items():
    assert final_by_applicant[feature].isna().eq(final_by_applicant[denominator].eq(0)).all(), feature
for source, label in [("AMT_PAYMENT_CURRENT", "current_payment"), ("AMT_PAYMENT_TOTAL_CURRENT", "total_payment")]:
    observed_latest = credit_card_grouped["latest_" + source].count().reindex(final_by_applicant.index, fill_value=0)
    assert final_by_applicant["credit_card_latest_" + label + "_mean"].isna().eq(observed_latest.eq(0)).all()


In [48]:
for prefix in DPD_FAMILIES.values():
    feature_prefix = "credit_card_" + prefix
    no_known = final_by_applicant[feature_prefix + "_observed_months"].eq(0)
    for suffix in ["_mean", "_max", "_positive_months", "_delinquent_contracts", "_repeated_contracts", "_ge90_contracts"]:
        assert final_by_applicant[feature_prefix + suffix].isna().eq(no_known).all(), feature_prefix + suffix
    assert final_by_applicant[feature_prefix + "_consecutive_contracts"].isna().eq(final_by_applicant[feature_prefix + "_consecutive_evaluable_contracts"].eq(0)).all()
    for suffix in ["_latest_mean", "_latest_max"]:
        assert final_by_applicant[feature_prefix + suffix].isna().eq(final_by_applicant[feature_prefix + "_latest_observed_contracts"].eq(0)).all()
    assert final_by_applicant[feature_prefix + "_last_positive_month"].isna().eq(final_by_applicant[feature_prefix + "_positive_months"].fillna(0).eq(0)).all()
    no_recent = final_by_applicant[feature_prefix + "_recent_6m_observed_months"].eq(0)
    for suffix in ["_recent_6m_positive_months", "_has_recent_6m_delinquency"]:
        assert final_by_applicant[feature_prefix + suffix].isna().eq(no_recent).all()
    assert final_by_applicant[feature_prefix + "_has_recent_6m_delinquency"].dropna().isin([0, 1]).all()
assert final_by_applicant["credit_card_has_recent_6m_high_utilization"].dropna().isin([0, 1]).all()
validation_results["Feature nulls agree with source, latest-state, utilization, DPD-pair, and recent availability"] = "PASS"
del final_by_applicant


## 12. Save the Applicant Features to Parquet

The output follows `data/interim/`, contains `SK_ID_CURR` plus numeric `credit_card_`
features, and has no saved index. We create a new file exclusively. On reruns an
identical existing output is checked and preserved; a different output stops for
review rather than being overwritten.


In [49]:
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
if OUTPUT_PATH.exists():
    existing_credit_card = pd.read_parquet(OUTPUT_PATH)
    pd.testing.assert_frame_equal(existing_credit_card, credit_card_features, check_exact=True)
    print("Existing identical output verified and preserved:", OUTPUT_PATH)
    del existing_credit_card
else:
    with OUTPUT_PATH.open("xb") as output_file:
        credit_card_features.to_parquet(output_file, index=False)
    print("Saved:", OUTPUT_PATH)


Saved: e:\Depi Notes Data Engineering\DEPI project R5\ProJect\data\interim\credit_card_features.parquet


## 13. Reload and Verify the Output

We compare shape, unique IDs, column order, dtypes, exact values, and nulls with
the in-memory feature table. Both raw input files are fingerprinted again.


In [50]:
saved_credit_card = pd.read_parquet(OUTPUT_PATH)
print("Reloaded shape:", saved_credit_card.shape)
print("Unique IDs:", saved_credit_card["SK_ID_CURR"].is_unique)
print("Duplicate IDs:", saved_credit_card["SK_ID_CURR"].duplicated().sum())
assert saved_credit_card.shape == credit_card_features.shape
assert saved_credit_card["SK_ID_CURR"].is_unique
assert saved_credit_card["SK_ID_CURR"].notna().all()
pd.testing.assert_frame_equal(saved_credit_card, credit_card_features, check_exact=True)
pd.testing.assert_series_equal(saved_credit_card.isna().sum(), credit_card_features.isna().sum())
validation_results["Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls"] = "PASS"
saved_credit_card.head()


Reloaded shape: (338857, 136)
Unique IDs: True
Duplicate IDs: 0


,SK_ID_CURR,credit_card_total_previous_contracts,credit_card_contracts_with_history,credit_card_latest_active_contracts,credit_card_latest_completed_contracts,credit_card_latest_other_status_contracts,credit_card_latest_unknown_status_contracts,credit_card_status_change_evaluable_contracts,credit_card_contracts_with_status_changes,credit_card_previous_contracts_without_card_history,...,credit_card_zero_credit_limit_months,credit_card_zero_credit_limit_contracts,credit_card_dpd_comparable_months,credit_card_dpd_disagreement_months,credit_card_dpd_disagreement_contracts,credit_card_current_payment_missing_months,credit_card_total_payment_missing_months,credit_card_minimum_instalment_missing_months,credit_card_drawings_missing_months,credit_card_utilization_missing_months
0,100001,1,0,0,0,0,0,0,NaN,1,...,0,0,0,NaN,NaN,0,0,0,0,0
1,100002,1,0,0,0,0,0,0,NaN,1,...,0,0,0,NaN,NaN,0,0,0,0,0
2,100003,3,0,0,0,0,0,0,NaN,3,...,0,0,0,NaN,NaN,0,0,0,0,0
3,100004,1,0,0,0,0,0,0,NaN,1,...,0,0,0,NaN,NaN,0,0,0,0,0
4,100005,2,0,0,0,0,0,0,NaN,2,...,0,0,0,NaN,NaN,0,0,0,0,0


In [51]:
for path, original_sha256 in raw_sha256_before.items():
    with path.open("rb") as raw_file:
        current_sha256 = hashlib.file_digest(raw_file, "sha256").hexdigest()
    assert current_sha256 == original_sha256, f"Raw input changed: {path.name}"
validation_results["Both raw CSV SHA-256 fingerprints unchanged"] = "PASS"
display(saved_credit_card.dtypes.value_counts().rename("columns").to_frame())
display(pd.Series(validation_results, name="Result").to_frame())


,columns
float64,65
int32,54
float32,17


,Result
"Source schema, IDs, relative time, unique monthly grain, and numeric domains",PASS
"Unambiguous mapping, source applicant audit, and unmatched contract/row conservation",PASS
"Monthly utilization eligibility, signed values, and above-limit ratios preserved",PASS
"Contract grain, monthly counts, independent gaps, actual latest rows, and status-pair coverage",PASS
"Amount observations/sums, negative balances, uncapped utilization, and DPD comparison conservation",PASS
"SK_DPD contract observation, positivity, severity, and consecutive checks",PASS
"SK_DPD_DEF contract observation, positivity, severity, and consecutive checks",PASS
"Direct contract source checks: latest values, history gaps, signed amounts, and utilization",PASS
"Direct ordinary/adjusted consecutive DPD, status-change, and relative recent-window checks",PASS
"Unique final applicant grain, exact mapping coverage, and card/month count conservation",PASS


## 14. Final Summary

This report uses statistics from this execution. Source observed applicants,
mapped card applicants, and the final previous-mapping population are separate
populations. Unmatched source histories remain inspectable at contract grain.


In [52]:
n_engineered_features = saved_credit_card.shape[1] - 1
category_summary = "\n".join(f"| {name} | {group.shape[1] - 1} |" for name, group in feature_groups.items())
validation_summary = "\n".join(f"| {check} | {result} |" for check, result in validation_results.items())
display(Markdown(f"""
| Summary | Result |
|---|---|
| Source dimensions | {source_shape[0]:,} monthly rows × {source_shape[1]} columns |
| Unique credit-card contracts | {n_source_contracts:,} |
| Matched / unmatched contracts | {len(matched_ids):,} / {len(unmatched_ids):,} |
| Contract ID match coverage | {len(matched_ids) / n_source_contracts:.2%} |
| Matched / unmatched monthly rows | {n_matched_rows:,} / {n_unmatched_rows:,} |
| Unmatched monthly-row share | {n_unmatched_rows / source_shape[0]:.2%} |
| Source observed applicants (audit only) | {f'{n_source_applicants:,}' if n_source_applicants is not None else 'Column absent'} |
| Applicants with mapped card history | {n_mapped_card_applicants:,} |
| Applicants in final mapping population | {n_applicants:,} |
| Applicants without mapped card history | {int(no_card_history.sum()):,} |
| Median observed months / contracts with gaps | {contract_features['observed_months'].median():.0f} / {int(contract_features['has_history_gap'].sum()):,} |
| Source contracts with observed status changes | {source_status_changes:,} |
| Source eligible utilization months | {source_utilization_audit['observed_months']:,} |
| Source >100% utilization months | {source_utilization_audit['over100_months']:,} |
| Latest source contracts with >100% utilization | {int(contract_features['latest_over100_utilization'].sum()):,} |
| Source negative balance / AMT_RECIVABLE observations | {source_money_audit['balance']['negative']:,} / {source_money_audit['recivable']['negative']:,} |
| Source current-payment histories with no known value | {int(contract_features['current_payment_observed_months'].eq(0).sum()):,} |
| Source total-drawing histories with no known value | {int(contract_features['drawings_observed_months'].eq(0).sum()):,} |
| Source ordinary / adjusted positive DPD months | {source_dpd_audit['dpd']['positive']:,} / {source_dpd_audit['dpd_def']['positive']:,} |
| Source DPD disagreement / adjusted-greater rows | {source_dpd_disagreement_rows:,} / {source_adjusted_greater_rows:,} |
| Source drawing amount mismatch rows / complete comparable rows | {source_drawing_audit['amount_mismatch_rows']:,} / {source_drawing_audit['complete_comparable_rows']:,} |
| Duplicate contract-month extras | {duplicate_key_extras:,} |
| Engineered features, excluding SK_ID_CURR | {n_engineered_features} |
| Final and reloaded shape | {saved_credit_card.shape} |
| Output | `{OUTPUT_PATH.relative_to(REPO_ROOT)}` |
| Raw input contents unchanged | Both SHA-256 fingerprints matched |

| Feature category | Features |
|---|---|
{category_summary}

| Validation | Result |
|---|---|
{validation_summary}
"""))
assert sum(group.shape[1] - 1 for group in feature_groups.values()) == n_engineered_features



| Summary | Result |
|---|---|
| Source dimensions | 3,840,312 monthly rows × 23 columns |
| Unique credit-card contracts | 104,307 |
| Matched / unmatched contracts | 92,935 / 11,372 |
| Contract ID match coverage | 89.10% |
| Matched / unmatched monthly rows | 2,757,496 / 1,082,816 |
| Unmatched monthly-row share | 28.20% |
| Source observed applicants (audit only) | 103,558 |
| Applicants with mapped card history | 92,447 |
| Applicants in final mapping population | 338,857 |
| Applicants without mapped card history | 246,410 |
| Median observed months / contracts with gaps | 21 / 14 |
| Source contracts with observed status changes | 17,888 |
| Source eligible utilization months | 3,086,489 |
| Source >100% utilization months | 334,362 |
| Latest source contracts with >100% utilization | 9,828 |
| Source negative balance / AMT_RECIVABLE observations | 2,345 / 109,338 |
| Source current-payment histories with no known value | 31,862 |
| Source total-drawing histories with no known value | 0 |
| Source ordinary / adjusted positive DPD months | 153,355 / 89,340 |
| Source DPD disagreement / adjusted-greater rows | 64,439 / 0 |
| Source drawing amount mismatch rows / complete comparable rows | 7,150 / 3,090,496 |
| Duplicate contract-month extras | 0 |
| Engineered features, excluding SK_ID_CURR | 135 |
| Final and reloaded shape | (338857, 136) |
| Output | `data\interim\credit_card_features.parquet` |
| Raw input contents unchanged | Both SHA-256 fingerprints matched |

| Feature category | Features |
|---|---|
| Contract counts, latest status, and coverage | 12 |
| History length and gaps | 10 |
| Historical and latest utilization | 13 |
| Balances, limits, and receivables | 12 |
| Payments, drawings, and descriptive ratios | 20 |
| Ordinary DPD | 16 |
| Adjusted DPD | 16 |
| Recent observations and DPD recency | 16 |
| Missingness and observed ledger states | 20 |

| Validation | Result |
|---|---|
| Source schema, IDs, relative time, unique monthly grain, and numeric domains | PASS |
| Unambiguous mapping, source applicant audit, and unmatched contract/row conservation | PASS |
| Monthly utilization eligibility, signed values, and above-limit ratios preserved | PASS |
| Contract grain, monthly counts, independent gaps, actual latest rows, and status-pair coverage | PASS |
| Amount observations/sums, negative balances, uncapped utilization, and DPD comparison conservation | PASS |
| SK_DPD contract observation, positivity, severity, and consecutive checks | PASS |
| SK_DPD_DEF contract observation, positivity, severity, and consecutive checks | PASS |
| Direct contract source checks: latest values, history gaps, signed amounts, and utilization | PASS |
| Direct ordinary/adjusted consecutive DPD, status-change, and relative recent-window checks | PASS |
| Unique final applicant grain, exact mapping coverage, and card/month count conservation | PASS |
| Every applicant latest-status bucket and detected status changes conserved | PASS |
| SK_DPD applicant positivity, recurrence, severity, and recent counts conserved | PASS |
| SK_DPD_DEF applicant positivity, recurrence, severity, and recent counts conserved | PASS |
| Negative ledger observations and mapped utilization maxima retained | PASS |
| Direct applicant source checks: weighted amounts, actual latest stock/payment/utilization, and recent means | PASS |
| Independent ordinary/adjusted applicant delay, recurrence, latest state, and recency checks | PASS |
| All documented eligible-rate formulas, 0–1 shares, and no unexpected infinities | PASS |
| Feature nulls agree with source, latest-state, utilization, DPD-pair, and recent availability | PASS |
| Parquet reload exactly preserves shape, IDs, dtypes, values, and nulls | PASS |
| Both raw CSV SHA-256 fingerprints unchanged | PASS |


### Decisions and Limitations

- **Mapping/population:** Only previous-application IDs supply feature applicant
  mappings. The source applicant ID is audited, then removed. Unmatched contracts
  remain in `unmatched_credit_card_contracts` and are excluded from applicant sums.
  Final rows cover all previous-mapping applicants. Mapping history coverage is
  not product-specific reporting completeness; applicants outside this mapping
  require a later join to the main applicant table.
- **Monthly history:** Latest means latest observed, not current-day state.
  Missing months are not inserted as zero activity. Span and observed-month counts
  remain distinct. Any repeated key stops execution for review, preserving raw inputs.
- **Utilization:** Positive finite limits and finite observed balances are required.
  Signed negative ratios and values above 100% remain unchanged. High-utilization
  thresholds are analytical, not confirmed default or causal indicators.
- **Amounts:** Negative ledger values stay signed and are separately counted.
  Historical stock sums are intermediate arithmetic for weighted means, not
  exposure features. Latest totals sum available last-card values and may combine
  different observation months; they are not guaranteed synchronized balances.
- **Payments/drawings:** Current and total payments retain different reporting
  scopes. Same-month payment/balance and payment/minimum ratios are descriptive,
  not billing compliance. Missing drawings/categories are not zero; complete
  category sums do not replace source drawings. `AMT_RECIVABLE` stays unchanged,
  and absent `AMT_TRANS_TOTAL` is never constructed.
- **Delinquency:** Ordinary and tolerance-adjusted DPD remain separate. Repeated
  positive months can be persistent arrears; consecutive flags require adjacent
  known months and retain pair availability. ≥90 is an analytical severity flag.
  Longer histories increase detection opportunity, not necessarily intrinsic risk.
- **Recent window:** −6 through −1 is a reviewable relative-month choice. Recent
  high utilization means ≥90%; no eligible recent evidence gives missing behavior.
  Offsets are not converted into calendar dates.
- **Availability/leakage:** Only explicit empty-set counts receive zero. Unknown
  measurements remain missing. All used months precede the application; production
  use still requires checking reporting latency and source availability. No target,
  target-derived transformation, or final application-table merge is used.
